VERSION 1

In [ ]:
# adaptation/few_shot_adapt.py

import os
import copy
import pandas as pd
import torch
import torch.optim as optim
from tqdm import tqdm

from config.base_config import cfg
from dataloaders.trajectory_datasets import TrajectoryDataset
from models.encoder import TrajEncoder
from models.neural_sde import NeuralSDE
from models.head import ForecastHead

# Reuse the simulator
from training.train_meta import simulate_neural_sde_batch

# --- EXPERIMENT CONFIG ---
ADAPT_STEPS = 50        # Fine-tuning steps
ADAPT_LR = 1e-2         # Aggressive learning rate for adaptation
SCRATCH_STEPS = 100     # Baseline training steps
SCRATCH_LR = 1e-2       # Baseline learning rate
OBS_LEN = 20            # The "Visible Past" (Blind the models to the future) before 50 now 20
N_SHOTS = 2             # Extreme Few-Shot setting

def get_task_data(dataset, theta_id, device):
    """Extracts all trajectories for a specific task ID."""
    rows = dataset.metadata[dataset.metadata["theta_id"] == theta_id]
    indices = rows.index.tolist()
    data = [dataset[i][0] for i in indices]
    return torch.stack(data).to(device)

def infer_z_star(encoder, support_trajs, obs_len):
    """Infers z* from the observed PAST of the support trajectories."""
    encoder.eval()
    with torch.no_grad():
        # Encoder only sees the first OBS_LEN steps
        obs = support_trajs[:, :obs_len, :]
        z_all = encoder(obs)
        z_mean = z_all.mean(dim=0, keepdim=True)
    return z_mean

def evaluate_model(sde, head, z, query_trajs, config, gen):
    """
    Evaluates on the FULL duration of Query trajectories.
    This tests Forecasting ability (Training on Past -> Predicting Future).
    """
    sde.eval()
    head.eval()
    
    T = config.time_grid.T
    n_steps = config.time_grid.n_steps
    x_max = config.stability.max_state_abs
    
    with torch.no_grad():
        x0 = query_trajs[:, 0, :]
        z_expanded = z.expand(x0.size(0), -1)
        
        # Simulate full duration
        traj_pred = simulate_neural_sde_batch(
            sde, x0, z_expanded, T, n_steps, x_max, gen
        )
        
        x_T_pred = traj_pred[:, -1, :]
        final_pred = head(x_T_pred, z_expanded)
        
        # Error over the FULL path (Past + Future)
        mse_path = torch.mean((traj_pred - query_trajs) ** 2).item()
        
        # Error at the final step (Long-term forecast)
        mse_head = torch.mean((final_pred - query_trajs[:, -1, :]) ** 2).item()
        
    return mse_path, mse_head

def fine_tune_head(sde, head, z_star, support_trajs, config, gen):
    """
    Fine-tunes Head ONLY on the observed PAST (first OBS_LEN steps).
    """
    head_ft = copy.deepcopy(head)
    optimizer = optim.Adam(head_ft.parameters(), lr=ADAPT_LR)
    
    head_ft.train()
    sde.eval()

    T = config.time_grid.T
    n_steps = config.time_grid.n_steps
    x_max = config.stability.max_state_abs
    
    # 1. Prepare Training Data (PAST ONLY)
    # We only train on the snippet we have seen
    obs_data = support_trajs[:, :OBS_LEN, :] 
    x0 = obs_data[:, 0, :]
    target_final_obs = obs_data[:, -1, :] # The state at step 50
    
    # We need to simulate only up to OBS_LEN for the loss
    # Calculate T_obs (time at step 50)
    dt = T / n_steps
    T_obs = dt * (OBS_LEN - 1) 
    
    z_expanded = z_star.expand(x0.size(0), -1)

    for _ in range(ADAPT_STEPS):
        optimizer.zero_grad()
        with torch.no_grad():
            # Simulate only up to the observed horizon
            traj_pred = simulate_neural_sde_batch(
                sde, x0, z_expanded, T_obs, OBS_LEN-1, x_max, gen
            )
            x_T_obs = traj_pred[:, -1, :]
            
        pred = head_ft(x_T_obs, z_expanded)
        loss = torch.nn.functional.mse_loss(pred, target_final_obs)
        loss.backward()
        optimizer.step()
        
    return head_ft

def train_from_scratch(x_dim, z_dim, support_trajs, config, gen):
    """
    Baseline: Trains from scratch BUT only on the observed PAST.
    Must forecast the unobserved future during evaluation.
    """
    from models.neural_sde import NeuralSDE
    from models.head import ForecastHead

    device = support_trajs.device
    sde_new = NeuralSDE(x_dim, z_dim, hidden_dim=32).to(device)
    head_new = ForecastHead(x_dim, z_dim, hidden_dim=32).to(device)
    z_zero = torch.zeros(1, z_dim, device=device)
    
    optimizer = optim.Adam(
        list(sde_new.parameters()) + list(head_new.parameters()),
        lr=SCRATCH_LR,
    )

    T = config.time_grid.T
    n_steps = config.time_grid.n_steps
    x_max = config.stability.max_state_abs
    
    # --- CRITICAL CHANGE: Only train on visible past ---
    obs_data = support_trajs[:, :OBS_LEN, :]
    x0 = obs_data[:, 0, :]
    
    # Simulation horizon for training is shortened
    dt = T / n_steps
    T_obs = dt * (OBS_LEN - 1)
    
    z_expanded = z_zero.expand(x0.size(0), -1)

    for _ in range(SCRATCH_STEPS):
        optimizer.zero_grad()
        
        # Simulate short path (Past)
        traj_pred = simulate_neural_sde_batch(
            sde_new, x0, z_expanded, T_obs, OBS_LEN-1, x_max, gen
        )
        pred_final = head_new(traj_pred[:, -1, :], z_expanded)
        
        # Loss only on observed data
        loss_path = torch.nn.functional.mse_loss(traj_pred, obs_data)
        loss_head = torch.nn.functional.mse_loss(pred_final, obs_data[:, -1, :])
        
        loss = loss_path + loss_head
        loss.backward()
        torch.nn.utils.clip_grad_norm_(sde_new.parameters(), 1.0)
        optimizer.step()
        
    return sde_new, head_new, z_zero

def main():
    device = torch.device(cfg.device)
    print(f"🔬 Starting Adaptation Experiment on {device}")
    print(f"   Config: {N_SHOTS}-Shot | OBS_LEN={OBS_LEN} (Forecasting Mode)")

    # Load Model
    ckpt_path = "checkpoints/meta_epoch_50.pt"
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found at {ckpt_path}")

    print(f"Loading checkpoint: {ckpt_path}")
    ckpt = torch.load(ckpt_path, map_location=device)
    
    x_dim = cfg.basis.x_dim
    z_dim = cfg.latent.latent_dim
    
    encoder = TrajEncoder(x_dim, z_dim, cfg.latent.encoder_hidden_dim, num_layers=2, dropout=0.1).to(device)
    sde = NeuralSDE(x_dim, z_dim, cfg.latent.sde_hidden_dim).to(device)
    head = ForecastHead(x_dim, z_dim, cfg.latent.head_hidden_dim).to(device)
    
    encoder.load_state_dict(ckpt['encoder'])
    sde.load_state_dict(ckpt['sde'])
    head.load_state_dict(ckpt['head'])
    
    gen = torch.Generator(device=device)
    gen.manual_seed(999)
    
    results_list = []
    regimes = ['testA', 'testB', 'testC']
    index_path = os.path.join(cfg.paths.data_root, "index.csv")
    
    for regime in regimes:
        print(f"\n--- Processing {regime} ---")
        try:
            ds_support = TrajectoryDataset(index_path, regime, "support", check_shapes=True)
            ds_query = TrajectoryDataset(index_path, regime, "query", check_shapes=True)
        except RuntimeError:
            print(f"Skipping {regime} (Empty)")
            continue
            
        tasks = ds_support.metadata['theta_id'].unique()
        
        for theta_id in tqdm(tasks, desc=f"Adapting {regime}"):
            # A. Get Data (2-Shot)
            full_support = get_task_data(ds_support, theta_id, device)
            support_trajs = full_support[:N_SHOTS]
            query_trajs = get_task_data(ds_query, theta_id, device)
            
            # B. Zero-Shot
            z_star = infer_z_star(encoder, support_trajs, OBS_LEN)
            zs_mse_path, zs_mse_head = evaluate_model(sde, head, z_star, query_trajs, cfg, gen)
            
            # C. Few-Shot
            head_ft = fine_tune_head(sde, head, z_star, support_trajs, cfg, gen)
            fs_mse_path, fs_mse_head = evaluate_model(sde, head_ft, z_star, query_trajs, cfg, gen)
            
            # D. From-Scratch (Forecast Mode)
            sde_scr, head_scr, z_scr = train_from_scratch(x_dim, z_dim, support_trajs, cfg, gen)
            scr_mse_path, scr_mse_head = evaluate_model(sde_scr, head_scr, z_scr, query_trajs, cfg, gen)
            
            results_list.append({
                "regime": regime,
                "theta_id": theta_id,
                "n_shots": N_SHOTS,
                "mse_path_zeroshot": zs_mse_path,
                "mse_path_fewshot": fs_mse_path,
                "mse_path_scratch": scr_mse_path,
            })
            
    # Save Results
    os.makedirs("results", exist_ok=True)
    df = pd.DataFrame(results_list)
    df.to_csv("results/adaptation_results.csv", index=False)
    
    print("\n✅ Forecasting Adaptation Complete!")
    print("Average MSE per Regime (Full Path Forecast):")
    print(df.groupby("regime")[["mse_path_zeroshot", "mse_path_fewshot", "mse_path_scratch"]].mean())

if __name__ == "__main__":
    main()

✅ Forecasting Adaptation Complete!
Average MSE per Regime (Full Path Forecast):
        mse_path_zeroshot  mse_path_fewshot  mse_path_scratch
regime                                                       
testA            0.334481          0.334488          0.319265
testB            0.911970          0.911916          0.835359
testC            1.336400          1.336430          0.858306


VERSION 2 

In [ ]:
# adaptation/few_shot_adapt.py

import os
import copy
import pandas as pd
import torch
import torch.optim as optim
from tqdm import tqdm

from config.base_config import cfg
from dataloaders.trajectory_datasets import TrajectoryDataset
from models.encoder import TrajEncoder
from models.neural_sde import NeuralSDE
from models.head import ForecastHead

# Reuse the simulator
from training.train_meta import simulate_neural_sde_batch

# --- EXPERIMENT CONFIG ---
ADAPT_STEPS = 50        # Fine-tuning steps for the Head
ADAPT_LR = 1e-2         # Aggressive learning rate for Head adaptation
SCRATCH_STEPS = 50      # ⬅️ CHANGED: Same budget as Meta (Fair comparison)
SCRATCH_LR = 1e-2       # Baseline learning rate
OBS_LEN = 20            # The "Visible Past" (Starve the baseline with only 20 steps)
N_SHOTS = 2             # Extreme Few-Shot setting

# --- Latent Optimization Config ---
REFINE_STEPS = 25       # How many steps to polish z
REFINE_LR = 0.1         # Learning rate for z optimization

def get_task_data(dataset, theta_id, device):
    """Extracts all trajectories for a specific task ID."""
    rows = dataset.metadata[dataset.metadata["theta_id"] == theta_id]
    indices = rows.index.tolist()
    data = [dataset[i][0] for i in indices]
    return torch.stack(data).to(device)

def infer_z_star(encoder, support_trajs, obs_len):
    """Infers initial z* from the observed PAST of the support trajectories."""
    encoder.eval()
    with torch.no_grad():
        # Encoder only sees the first OBS_LEN steps
        obs = support_trajs[:, :obs_len, :]
        z_all = encoder(obs)
        z_mean = z_all.mean(dim=0, keepdim=True)
    return z_mean

def refine_z(sde, z_init, support_data, config, gen, steps=20, lr=0.1):
    """
    Performs Test-Time Optimization on z.
    We freeze the SDE and only tune z to match the observed history.
    """
    # 1. Create a trainable copy of z
    z_opt = z_init.clone().detach().requires_grad_(True)
    
    # 2. Optimizer specifically for z
    optimizer = optim.Adam([z_opt], lr=lr)
    
    # 3. Setup Simulation Parameters for the Short History
    T = config.time_grid.T
    n_steps = config.time_grid.n_steps
    dt = T / n_steps
    
    # We only simulate up to OBS_LEN
    obs_len = support_data.shape[1] 
    T_obs = dt * (obs_len - 1)
    
    x0 = support_data[:, 0, :]
    x_max = config.stability.max_state_abs

    # 4. Optimization Loop
    for i in range(steps):
        optimizer.zero_grad()
        
        # Expand z for the batch
        z_expanded = z_opt.expand(x0.size(0), -1)
        
        # Simulate ONLY the past
        traj_pred = simulate_neural_sde_batch(
            sde, x0, z_expanded, T_obs, obs_len - 1, x_max, gen
        )
        
        # MSE Loss on the observed trajectory
        loss = torch.nn.functional.mse_loss(traj_pred, support_data)
        
        loss.backward()
        optimizer.step()
        
    return z_opt.detach() # Return the fixed, optimized code

def evaluate_model(sde, head, z, query_trajs, config, gen):
    """
    Evaluates on the FULL duration of Query trajectories.
    Returns both Path MSE (Physics) and Head MSE (Forecast).
    """
    sde.eval()
    head.eval()
    
    T = config.time_grid.T
    n_steps = config.time_grid.n_steps
    x_max = config.stability.max_state_abs
    
    with torch.no_grad():
        x0 = query_trajs[:, 0, :]
        z_expanded = z.expand(x0.size(0), -1)
        
        # Simulate full duration
        traj_pred = simulate_neural_sde_batch(
            sde, x0, z_expanded, T, n_steps, x_max, gen
        )
        
        x_T_pred = traj_pred[:, -1, :]
        final_pred = head(x_T_pred, z_expanded)
        
        # Error over the FULL path (Physics/Dynamics Quality)
        mse_path = torch.mean((traj_pred - query_trajs) ** 2).item()
        
        # Error at the final step (Forecasting Quality)
        mse_head = torch.mean((final_pred - query_trajs[:, -1, :]) ** 2).item()
        
    return mse_path, mse_head

def fine_tune_head(sde, head, z_star, support_trajs, config, gen):
    """
    Fine-tunes Head ONLY on the observed PAST.
    """
    head_ft = copy.deepcopy(head)
    optimizer = optim.Adam(head_ft.parameters(), lr=ADAPT_LR)
    
    head_ft.train()
    sde.eval()

    T = config.time_grid.T
    n_steps = config.time_grid.n_steps
    x_max = config.stability.max_state_abs
    
    # Prepare Training Data (PAST ONLY)
    obs_data = support_trajs[:, :OBS_LEN, :] 
    x0 = obs_data[:, 0, :]
    target_final_obs = obs_data[:, -1, :] 
    
    dt = T / n_steps
    T_obs = dt * (OBS_LEN - 1) 
    
    z_expanded = z_star.expand(x0.size(0), -1)

    for _ in range(ADAPT_STEPS):
        optimizer.zero_grad()
        with torch.no_grad():
            traj_pred = simulate_neural_sde_batch(
                sde, x0, z_expanded, T_obs, OBS_LEN-1, x_max, gen
            )
            x_T_obs = traj_pred[:, -1, :]
            
        pred = head_ft(x_T_obs, z_expanded)
        loss = torch.nn.functional.mse_loss(pred, target_final_obs)
        loss.backward()
        optimizer.step()
        
    return head_ft

def train_from_scratch(x_dim, z_dim, support_trajs, config, gen):
    """
    Baseline: Trains from scratch BUT only on the observed PAST.
    """
    from models.neural_sde import NeuralSDE
    from models.head import ForecastHead

    device = support_trajs.device
    sde_new = NeuralSDE(x_dim, z_dim, hidden_dim=32).to(device)
    head_new = ForecastHead(x_dim, z_dim, hidden_dim=32).to(device)
    z_zero = torch.zeros(1, z_dim, device=device)
    
    optimizer = optim.Adam(
        list(sde_new.parameters()) + list(head_new.parameters()),
        lr=SCRATCH_LR,
    )

    T = config.time_grid.T
    n_steps = config.time_grid.n_steps
    x_max = config.stability.max_state_abs
    
    # --- CRITICAL: Only train on visible past ---
    obs_data = support_trajs[:, :OBS_LEN, :]
    x0 = obs_data[:, 0, :]
    
    dt = T / n_steps
    T_obs = dt * (OBS_LEN - 1)
    
    z_expanded = z_zero.expand(x0.size(0), -1)

    # ⬅️ Uses SCRATCH_STEPS (now 50, same as Meta)
    for _ in range(SCRATCH_STEPS):
        optimizer.zero_grad()
        
        # Simulate short path (Past)
        traj_pred = simulate_neural_sde_batch(
            sde_new, x0, z_expanded, T_obs, OBS_LEN-1, x_max, gen
        )
        pred_final = head_new(traj_pred[:, -1, :], z_expanded)
        
        # Loss only on observed data
        loss_path = torch.nn.functional.mse_loss(traj_pred, obs_data)
        loss_head = torch.nn.functional.mse_loss(pred_final, obs_data[:, -1, :])
        
        loss = loss_path + loss_head
        loss.backward()
        torch.nn.utils.clip_grad_norm_(sde_new.parameters(), 1.0)
        optimizer.step()
        
    return sde_new, head_new, z_zero

def main():
    device = torch.device(cfg.device)
    print(f"🔬 Starting Adaptation Experiment on {device}")
    print(f"   Config: {N_SHOTS}-Shot | OBS_LEN={OBS_LEN}")
    print(f"   Steps: Meta={ADAPT_STEPS} vs Scratch={SCRATCH_STEPS} (Fair Budget)")
    print(f"   Latent Refine: Steps={REFINE_STEPS}, LR={REFINE_LR}")

    # Load Model
    ckpt_path = "checkpoints/meta_epoch_50.pt"
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found at {ckpt_path}")

    print(f"Loading checkpoint: {ckpt_path}")
    ckpt = torch.load(ckpt_path, map_location=device)
    
    x_dim = cfg.basis.x_dim
    z_dim = cfg.latent.latent_dim
    
    encoder = TrajEncoder(x_dim, z_dim, cfg.latent.encoder_hidden_dim, num_layers=2, dropout=0.1).to(device)
    sde = NeuralSDE(x_dim, z_dim, cfg.latent.sde_hidden_dim).to(device)
    head = ForecastHead(x_dim, z_dim, cfg.latent.head_hidden_dim).to(device)
    
    encoder.load_state_dict(ckpt['encoder'])
    sde.load_state_dict(ckpt['sde'])
    head.load_state_dict(ckpt['head'])
    
    gen = torch.Generator(device=device)
    gen.manual_seed(999)
    
    results_list = []
    regimes = ['testA', 'testB', 'testC']
    index_path = os.path.join(cfg.paths.data_root, "index.csv")
    
    for regime in regimes:
        print(f"\n--- Processing {regime} ---")
        try:
            ds_support = TrajectoryDataset(index_path, regime, "support", check_shapes=True)
            ds_query = TrajectoryDataset(index_path, regime, "query", check_shapes=True)
        except RuntimeError:
            print(f"Skipping {regime} (Empty)")
            continue
            
        tasks = ds_support.metadata['theta_id'].unique()
        
        for theta_id in tqdm(tasks, desc=f"Adapting {regime}"):
            full_support = get_task_data(ds_support, theta_id, device)
            support_trajs = full_support[:N_SHOTS]
            query_trajs = get_task_data(ds_query, theta_id, device)
            
            # B. Zero-Shot WITH Refinement
            z_init = infer_z_star(encoder, support_trajs, OBS_LEN)
            obs_data = support_trajs[:, :OBS_LEN, :]
            z_refined = refine_z(sde, z_init, obs_data, cfg, gen, steps=REFINE_STEPS, lr=REFINE_LR)
            
            zs_mse_path, zs_mse_head = evaluate_model(sde, head, z_refined, query_trajs, cfg, gen)
            
            # C. Few-Shot (Using the Refined z)
            head_ft = fine_tune_head(sde, head, z_refined, support_trajs, cfg, gen)
            fs_mse_path, fs_mse_head = evaluate_model(sde, head_ft, z_refined, query_trajs, cfg, gen)
            
            # D. From-Scratch (Forecast Mode)
            sde_scr, head_scr, z_scr = train_from_scratch(x_dim, z_dim, support_trajs, cfg, gen)
            scr_mse_path, scr_mse_head = evaluate_model(sde_scr, head_scr, z_scr, query_trajs, cfg, gen)
            
            # ⬅️ NEW: Saving Head Errors too
            results_list.append({
                "regime": regime,
                "theta_id": theta_id,
                "n_shots": N_SHOTS,
                
                # Path Errors
                "mse_path_zeroshot": zs_mse_path,
                "mse_path_fewshot": fs_mse_path,
                "mse_path_scratch": scr_mse_path,
                
                # Head Errors (Forecast)
                "mse_head_zeroshot": zs_mse_head,
                "mse_head_fewshot": fs_mse_head,
                "mse_head_scratch": scr_mse_head,
            })
            
    # Save Results
    os.makedirs("results", exist_ok=True)
    df = pd.DataFrame(results_list)
    df.to_csv("results/adaptation_results.csv", index=False)
    
    print("\n✅ Forecasting Adaptation Complete!")
    
    print("\nAverage PATH MSE (Full Trajectory Physics):")
    print(df.groupby("regime")[["mse_path_zeroshot", "mse_path_fewshot", "mse_path_scratch"]].mean())
    
    print("\nAverage FINAL-STEP MSE (Forecasting):")
    print(df.groupby("regime")[["mse_head_zeroshot", "mse_head_fewshot", "mse_head_scratch"]].mean())

if __name__ == "__main__":
    main()

✅ Forecasting Adaptation Complete!

Average PATH MSE (Full Trajectory Physics):
        mse_path_zeroshot  mse_path_fewshot  mse_path_scratch
regime                                                       
testA            0.291004          0.291125          0.333161
testB            0.550816          0.550851          0.733791
testC            0.751847          0.751888          0.832700

Average FINAL-STEP MSE (Forecasting):
        mse_head_zeroshot  mse_head_fewshot  mse_head_scratch
regime                                                       
testA            1.312699          0.677216          0.682977
testB            2.085251          2.360096          2.917257
testC            2.691119          3.481056          4.264273

VERSION 3, , 5 dimensions

In [ ]:
# adaptation/few_shot_adapt.py
# FIXED VERSION - Train head on FULL trajectory, not just the past

import os
import copy
import pandas as pd
import torch
import torch.optim as optim
from tqdm import tqdm

from config.base_config import cfg
from dataloaders.trajectory_datasets import TrajectoryDataset
from models.encoder import TrajEncoder
from models.neural_sde import NeuralSDE
from models.head import ForecastHead

# Reuse the simulator
from training.train_meta import simulate_neural_sde_batch

# --- EXPERIMENT CONFIG ---
ADAPT_STEPS = 50        # Fine-tuning steps for the Head
ADAPT_LR = 1e-2         # Aggressive learning rate for Head adaptation
SCRATCH_STEPS = 50      # Same budget as Meta (Fair comparison)
SCRATCH_LR = 1e-2       # Baseline learning rate
OBS_LEN = 20            # The "Visible Past" (Starve the baseline with only 20 steps)
N_SHOTS = 2             # Extreme Few-Shot setting

# --- Latent Optimization Config ---
REFINE_STEPS = 25       # How many steps to polish z
REFINE_LR = 0.1         # Learning rate for z optimization

def get_task_data(dataset, theta_id, device):
    """Extracts all trajectories for a specific task ID."""
    rows = dataset.metadata[dataset.metadata["theta_id"] == theta_id]
    indices = rows.index.tolist()
    data = [dataset[i][0] for i in indices]
    return torch.stack(data).to(device)

def infer_z_star(encoder, support_trajs, obs_len):
    """Infers initial z* from the observed PAST of the support trajectories."""
    encoder.eval()
    with torch.no_grad():
        # Encoder only sees the first OBS_LEN steps
        obs = support_trajs[:, :obs_len, :]
        z_all = encoder(obs)
        z_mean = z_all.mean(dim=0, keepdim=True)
    return z_mean

def refine_z(sde, z_init, support_data, config, gen, steps=20, lr=0.1):
    """
    Performs Test-Time Optimization on z.
    We freeze the SDE and only tune z to match the observed history.
    """
    # 1. Create a trainable copy of z
    z_opt = z_init.clone().detach().requires_grad_(True)
    
    # 2. Optimizer specifically for z
    optimizer = optim.Adam([z_opt], lr=lr)
    
    # 3. Setup Simulation Parameters for the Short History
    T = config.time_grid.T
    n_steps = config.time_grid.n_steps
    dt = T / n_steps
    
    # We only simulate up to OBS_LEN
    obs_len = support_data.shape[1] 
    T_obs = dt * (obs_len - 1)
    
    x0 = support_data[:, 0, :]
    x_max = config.stability.max_state_abs

    # 4. Optimization Loop
    for i in range(steps):
        optimizer.zero_grad()
        
        # Expand z for the batch
        z_expanded = z_opt.expand(x0.size(0), -1)
        
        # Simulate ONLY the past
        traj_pred = simulate_neural_sde_batch(
            sde, x0, z_expanded, T_obs, obs_len - 1, x_max, gen
        )
        
        # MSE Loss on the observed trajectory
        loss = torch.nn.functional.mse_loss(traj_pred, support_data)
        
        loss.backward()
        optimizer.step()
        
    return z_opt.detach() # Return the fixed, optimized code

def evaluate_model(sde, head, z, query_trajs, config, gen):
    """
    Evaluates on the FULL duration of Query trajectories.
    Returns both Path MSE (Physics) and Head MSE (Forecast).
    """
    sde.eval()
    head.eval()
    
    T = config.time_grid.T
    n_steps = config.time_grid.n_steps
    x_max = config.stability.max_state_abs
    
    with torch.no_grad():
        x0 = query_trajs[:, 0, :]
        z_expanded = z.expand(x0.size(0), -1)
        
        # Simulate full duration
        traj_pred = simulate_neural_sde_batch(
            sde, x0, z_expanded, T, n_steps, x_max, gen
        )
        
        x_T_pred = traj_pred[:, -1, :]
        final_pred = head(x_T_pred, z_expanded)
        
        # Error over the FULL path (Physics/Dynamics Quality)
        mse_path = torch.mean((traj_pred - query_trajs) ** 2).item()
        
        # Error at the final step (Forecasting Quality)
        mse_head = torch.mean((final_pred - query_trajs[:, -1, :]) ** 2).item()
        
    return mse_path, mse_head

def fine_tune_head(sde, head, z_star, support_trajs, config, gen):
    """
    Fine-tunes Head on the FULL trajectory.
    This ensures head trains on the same task it will be evaluated on.
    
    KEY CHANGE: Before, we trained on the first OBS_LEN steps only.
    Now we train on the full trajectory to avoid train-test mismatch.
    """
    head_ft = copy.deepcopy(head)
    optimizer = optim.Adam(head_ft.parameters(), lr=ADAPT_LR)
    
    head_ft.train()
    sde.eval()

    T = config.time_grid.T
    n_steps = config.time_grid.n_steps
    x_max = config.stability.max_state_abs
    
    # Use FULL trajectory for training (not just past)
    x0 = support_trajs[:, 0, :]
    target_final = support_trajs[:, -1, :]  # Target: final step of full trajectory
    
    z_expanded = z_star.expand(x0.size(0), -1)

    for _ in range(ADAPT_STEPS):
        optimizer.zero_grad()
        with torch.no_grad():
            # Simulate FULL duration (not truncated to OBS_LEN)
            traj_pred = simulate_neural_sde_batch(
                sde, x0, z_expanded, T, n_steps, x_max, gen
            )
            x_T_pred = traj_pred[:, -1, :]
            
        pred = head_ft(x_T_pred, z_expanded)
        loss = torch.nn.functional.mse_loss(pred, target_final)
        loss.backward()
        optimizer.step()
        
    return head_ft

def train_from_scratch(x_dim, z_dim, support_trajs, config, gen):
    """
    Baseline: Trains from scratch BUT only on the observed PAST.
    """
    from models.neural_sde import NeuralSDE
    from models.head import ForecastHead

    device = support_trajs.device
    sde_new = NeuralSDE(x_dim, z_dim, hidden_dim=32).to(device)
    head_new = ForecastHead(x_dim, z_dim, hidden_dim=32).to(device)
    z_zero = torch.zeros(1, z_dim, device=device)
    
    optimizer = optim.Adam(
        list(sde_new.parameters()) + list(head_new.parameters()),
        lr=SCRATCH_LR,
    )

    T = config.time_grid.T
    n_steps = config.time_grid.n_steps
    x_max = config.stability.max_state_abs
    
    # --- CRITICAL: Only train on visible past ---
    obs_data = support_trajs[:, :OBS_LEN, :]
    x0 = obs_data[:, 0, :]
    
    dt = T / n_steps
    T_obs = dt * (OBS_LEN - 1)
    
    z_expanded = z_zero.expand(x0.size(0), -1)

    # Uses SCRATCH_STEPS (now 50, same as Meta)
    for _ in range(SCRATCH_STEPS):
        optimizer.zero_grad()
        
        # Simulate short path (Past)
        traj_pred = simulate_neural_sde_batch(
            sde_new, x0, z_expanded, T_obs, OBS_LEN-1, x_max, gen
        )
        pred_final = head_new(traj_pred[:, -1, :], z_expanded)
        
        # Loss only on observed data
        loss_path = torch.nn.functional.mse_loss(traj_pred, obs_data)
        loss_head = torch.nn.functional.mse_loss(pred_final, obs_data[:, -1, :])
        
        loss = loss_path + loss_head
        loss.backward()
        torch.nn.utils.clip_grad_norm_(sde_new.parameters(), 1.0)
        optimizer.step()
        
    return sde_new, head_new, z_zero

def main():
    device = torch.device(cfg.device)
    print(f"🔬 Starting Adaptation Experiment on {device}")
    print(f"   Config: {N_SHOTS}-Shot | OBS_LEN={OBS_LEN}")
    print(f"   Steps: Meta={ADAPT_STEPS} vs Scratch={SCRATCH_STEPS} (Fair Budget)")
    print(f"   Latent Refine: Steps={REFINE_STEPS}, LR={REFINE_LR}")

    # Load Model
    ckpt_path = "checkpoints/meta_epoch_50.pt"
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found at {ckpt_path}")

    print(f"Loading checkpoint: {ckpt_path}")
    ckpt = torch.load(ckpt_path, map_location=device)
    
    x_dim = cfg.basis.x_dim
    z_dim = cfg.latent.latent_dim
    
    encoder = TrajEncoder(x_dim, z_dim, cfg.latent.encoder_hidden_dim, num_layers=2, dropout=0.1).to(device)
    sde = NeuralSDE(x_dim, z_dim, cfg.latent.sde_hidden_dim).to(device)
    head = ForecastHead(x_dim, z_dim, cfg.latent.head_hidden_dim).to(device)
    
    encoder.load_state_dict(ckpt['encoder'])
    sde.load_state_dict(ckpt['sde'])
    head.load_state_dict(ckpt['head'])
    
    gen = torch.Generator(device=device)
    gen.manual_seed(999)
    
    results_list = []
    regimes = ['testA', 'testB', 'testC']
    index_path = os.path.join(cfg.paths.data_root, "index.csv")
    
    for regime in regimes:
        print(f"\n--- Processing {regime} ---")
        try:
            ds_support = TrajectoryDataset(index_path, regime, "support", check_shapes=True)
            ds_query = TrajectoryDataset(index_path, regime, "query", check_shapes=True)
        except RuntimeError:
            print(f"Skipping {regime} (Empty)")
            continue
            
        tasks = ds_support.metadata['theta_id'].unique()
        
        for theta_id in tqdm(tasks, desc=f"Adapting {regime}"):
            full_support = get_task_data(ds_support, theta_id, device)
            support_trajs = full_support[:N_SHOTS]
            query_trajs = get_task_data(ds_query, theta_id, device)
            
            # B. Zero-Shot WITH Refinement
            z_init = infer_z_star(encoder, support_trajs, OBS_LEN)
            obs_data = support_trajs[:, :OBS_LEN, :]
            z_refined = refine_z(sde, z_init, obs_data, cfg, gen, steps=REFINE_STEPS, lr=REFINE_LR)
            
            zs_mse_path, zs_mse_head = evaluate_model(sde, head, z_refined, query_trajs, cfg, gen)
            
            # C. Few-Shot (Using the Refined z) - NOW WITH FIXED HEAD FINE-TUNING
            head_ft = fine_tune_head(sde, head, z_refined, support_trajs, cfg, gen)
            fs_mse_path, fs_mse_head = evaluate_model(sde, head_ft, z_refined, query_trajs, cfg, gen)
            
            # D. From-Scratch (Forecast Mode)
            sde_scr, head_scr, z_scr = train_from_scratch(x_dim, z_dim, support_trajs, cfg, gen)
            scr_mse_path, scr_mse_head = evaluate_model(sde_scr, head_scr, z_scr, query_trajs, cfg, gen)
            
            # Save results
            results_list.append({
                "regime": regime,
                "theta_id": theta_id,
                "n_shots": N_SHOTS,
                
                # Path Errors
                "mse_path_zeroshot": zs_mse_path,
                "mse_path_fewshot": fs_mse_path,
                "mse_path_scratch": scr_mse_path,
                
                # Head Errors (Forecast)
                "mse_head_zeroshot": zs_mse_head,
                "mse_head_fewshot": fs_mse_head,
                "mse_head_scratch": scr_mse_head,
            })
            
    # Save Results
    os.makedirs("results", exist_ok=True)
    df = pd.DataFrame(results_list)
    df.to_csv("results/adaptation_results.csv", index=False)
    
    print("\n✅ Forecasting Adaptation Complete!")
    
    print("\nAverage PATH MSE (Full Trajectory Physics):")
    print(df.groupby("regime")[["mse_path_zeroshot", "mse_path_fewshot", "mse_path_scratch"]].mean())
    
    print("\nAverage FINAL-STEP MSE (Forecasting):")
    print(df.groupby("regime")[["mse_head_zeroshot", "mse_head_fewshot", "mse_head_scratch"]].mean())

if __name__ == "__main__":
    main()

✅ Forecasting Adaptation Complete!

Average PATH MSE (Full Trajectory Physics):
        mse_path_zeroshot  mse_path_fewshot  mse_path_scratch
regime                                                       
testA            0.291756          0.291851          0.335807
testB            0.550988          0.551112          0.770010
testC            0.751939          0.751714          0.852782

Average FINAL-STEP MSE (Forecasting):
        mse_head_zeroshot  mse_head_fewshot  mse_head_scratch
regime                                                       
testA            1.319719          0.556532          0.728225
testB            2.086427          1.221991          2.803850
testC            2.693532          1.296466          4.266486

VERSION 4, 10 dimensions 

In [ ]:
# adaptation/few_shot_adapt.py
# FIXED VERSION - Train head on FULL trajectory, not just the past

import os
import copy
import pandas as pd
import torch
import torch.optim as optim
from tqdm import tqdm

from config.base_config import cfg
from dataloaders.trajectory_datasets import TrajectoryDataset
from models.encoder import TrajEncoder
from models.neural_sde import NeuralSDE
from models.head import ForecastHead

# Reuse the simulator
from training.train_meta import simulate_neural_sde_batch

# --- EXPERIMENT CONFIG ---
ADAPT_STEPS = 50        # Fine-tuning steps for the Head
ADAPT_LR = 1e-2         # Aggressive learning rate for Head adaptation
SCRATCH_STEPS = 50      # Same budget as Meta (Fair comparison)
SCRATCH_LR = 1e-2       # Baseline learning rate
OBS_LEN = 20            # The "Visible Past" (Starve the baseline with only 20 steps)
N_SHOTS = 2             # Extreme Few-Shot setting

# --- Latent Optimization Config ---
REFINE_STEPS = 25       # How many steps to polish z
REFINE_LR = 0.1         # Learning rate for z optimization

def get_task_data(dataset, theta_id, device):
    """Extracts all trajectories for a specific task ID."""
    rows = dataset.metadata[dataset.metadata["theta_id"] == theta_id]
    indices = rows.index.tolist()
    data = [dataset[i][0] for i in indices]
    return torch.stack(data).to(device)

def infer_z_star(encoder, support_trajs, obs_len):
    """Infers initial z* from the observed PAST of the support trajectories."""
    encoder.eval()
    with torch.no_grad():
        # Encoder only sees the first OBS_LEN steps
        obs = support_trajs[:, :obs_len, :]
        z_all = encoder(obs)
        z_mean = z_all.mean(dim=0, keepdim=True)
    return z_mean

def refine_z(sde, z_init, support_data, config, gen, steps=20, lr=0.1):
    """
    Performs Test-Time Optimization on z.
    We freeze the SDE and only tune z to match the observed history.
    """
    # 1. Create a trainable copy of z
    z_opt = z_init.clone().detach().requires_grad_(True)
    
    # 2. Optimizer specifically for z
    optimizer = optim.Adam([z_opt], lr=lr)
    
    # 3. Setup Simulation Parameters for the Short History
    T = config.time_grid.T
    n_steps = config.time_grid.n_steps
    dt = T / n_steps
    
    # We only simulate up to OBS_LEN
    obs_len = support_data.shape[1] 
    T_obs = dt * (obs_len - 1)
    
    x0 = support_data[:, 0, :]
    x_max = config.stability.max_state_abs

    # 4. Optimization Loop
    for i in range(steps):
        optimizer.zero_grad()
        
        # Expand z for the batch
        z_expanded = z_opt.expand(x0.size(0), -1)
        
        # Simulate ONLY the past
        traj_pred = simulate_neural_sde_batch(
            sde, x0, z_expanded, T_obs, obs_len - 1, x_max, gen
        )
        
        # MSE Loss on the observed trajectory
        loss = torch.nn.functional.mse_loss(traj_pred, support_data)
        
        loss.backward()
        optimizer.step()
        
    return z_opt.detach() # Return the fixed, optimized code

def evaluate_model(sde, head, z, query_trajs, config, gen):
    """
    Evaluates on the FULL duration of Query trajectories.
    Returns both Path MSE (Physics) and Head MSE (Forecast).
    """
    sde.eval()
    head.eval()
    
    T = config.time_grid.T
    n_steps = config.time_grid.n_steps
    x_max = config.stability.max_state_abs
    
    with torch.no_grad():
        x0 = query_trajs[:, 0, :]
        z_expanded = z.expand(x0.size(0), -1)
        
        # Simulate full duration
        traj_pred = simulate_neural_sde_batch(
            sde, x0, z_expanded, T, n_steps, x_max, gen
        )
        
        x_T_pred = traj_pred[:, -1, :]
        final_pred = head(x_T_pred, z_expanded)
        
        # Error over the FULL path (Physics/Dynamics Quality)
        mse_path = torch.mean((traj_pred - query_trajs) ** 2).item()
        
        # Error at the final step (Forecasting Quality)
        mse_head = torch.mean((final_pred - query_trajs[:, -1, :]) ** 2).item()
        
    return mse_path, mse_head

def fine_tune_head(sde, head, z_star, support_trajs, config, gen):
    """
    Fine-tunes Head on the FULL trajectory.
    This ensures head trains on the same task it will be evaluated on.
    
    KEY CHANGE: Before, we trained on the first OBS_LEN steps only.
    Now we train on the full trajectory to avoid train-test mismatch.
    """
    head_ft = copy.deepcopy(head)
    optimizer = optim.Adam(head_ft.parameters(), lr=ADAPT_LR)
    
    head_ft.train()
    sde.eval()

    T = config.time_grid.T
    n_steps = config.time_grid.n_steps
    x_max = config.stability.max_state_abs
    
    # Use FULL trajectory for training (not just past)
    x0 = support_trajs[:, 0, :]
    target_final = support_trajs[:, -1, :]  # Target: final step of full trajectory
    
    z_expanded = z_star.expand(x0.size(0), -1)

    for _ in range(ADAPT_STEPS):
        optimizer.zero_grad()
        with torch.no_grad():
            # Simulate FULL duration (not truncated to OBS_LEN)
            traj_pred = simulate_neural_sde_batch(
                sde, x0, z_expanded, T, n_steps, x_max, gen
            )
            x_T_pred = traj_pred[:, -1, :]
            
        pred = head_ft(x_T_pred, z_expanded)
        loss = torch.nn.functional.mse_loss(pred, target_final)
        loss.backward()
        optimizer.step()
        
    return head_ft

def train_from_scratch(x_dim, z_dim, support_trajs, config, gen):
    """
    Baseline: Trains from scratch BUT only on the observed PAST.
    """
    from models.neural_sde import NeuralSDE
    from models.head import ForecastHead

    device = support_trajs.device
    sde_new = NeuralSDE(x_dim, z_dim, hidden_dim=32).to(device)
    head_new = ForecastHead(x_dim, z_dim, hidden_dim=32).to(device)
    z_zero = torch.zeros(1, z_dim, device=device)
    
    optimizer = optim.Adam(
        list(sde_new.parameters()) + list(head_new.parameters()),
        lr=SCRATCH_LR,
    )

    T = config.time_grid.T
    n_steps = config.time_grid.n_steps
    x_max = config.stability.max_state_abs
    
    # --- CRITICAL: Only train on visible past ---
    obs_data = support_trajs[:, :OBS_LEN, :]
    x0 = obs_data[:, 0, :]
    
    dt = T / n_steps
    T_obs = dt * (OBS_LEN - 1)
    
    z_expanded = z_zero.expand(x0.size(0), -1)

    # Uses SCRATCH_STEPS (now 50, same as Meta)
    for _ in range(SCRATCH_STEPS):
        optimizer.zero_grad()
        
        # Simulate short path (Past)
        traj_pred = simulate_neural_sde_batch(
            sde_new, x0, z_expanded, T_obs, OBS_LEN-1, x_max, gen
        )
        pred_final = head_new(traj_pred[:, -1, :], z_expanded)
        
        # Loss only on observed data
        loss_path = torch.nn.functional.mse_loss(traj_pred, obs_data)
        loss_head = torch.nn.functional.mse_loss(pred_final, obs_data[:, -1, :])
        
        loss = loss_path + loss_head
        loss.backward()
        torch.nn.utils.clip_grad_norm_(sde_new.parameters(), 1.0)
        optimizer.step()
        
    return sde_new, head_new, z_zero

def main():
    device = torch.device(cfg.device)
    print(f"🔬 Starting Adaptation Experiment on {device}")
    print(f"   Config: {N_SHOTS}-Shot | OBS_LEN={OBS_LEN}")
    print(f"   Steps: Meta={ADAPT_STEPS} vs Scratch={SCRATCH_STEPS} (Fair Budget)")
    print(f"   Latent Refine: Steps={REFINE_STEPS}, LR={REFINE_LR}")

    # Load Model
    ckpt_path = "checkpoints/meta_epoch_50.pt"
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found at {ckpt_path}")

    print(f"Loading checkpoint: {ckpt_path}")
    ckpt = torch.load(ckpt_path, map_location=device)
    
    x_dim = cfg.basis.x_dim
    z_dim = cfg.latent.latent_dim
    
    encoder = TrajEncoder(x_dim, z_dim, cfg.latent.encoder_hidden_dim, num_layers=2, dropout=0.1).to(device)
    sde = NeuralSDE(x_dim, z_dim, cfg.latent.sde_hidden_dim).to(device)
    head = ForecastHead(x_dim, z_dim, cfg.latent.head_hidden_dim).to(device)
    
    encoder.load_state_dict(ckpt['encoder'])
    sde.load_state_dict(ckpt['sde'])
    head.load_state_dict(ckpt['head'])
    
    gen = torch.Generator(device=device)
    gen.manual_seed(999)
    
    results_list = []
    regimes = ['testA', 'testB', 'testC']
    index_path = os.path.join(cfg.paths.data_root, "index.csv")
    
    for regime in regimes:
        print(f"\n--- Processing {regime} ---")
        try:
            ds_support = TrajectoryDataset(index_path, regime, "support", check_shapes=True)
            ds_query = TrajectoryDataset(index_path, regime, "query", check_shapes=True)
        except RuntimeError:
            print(f"Skipping {regime} (Empty)")
            continue
            
        tasks = ds_support.metadata['theta_id'].unique()
        
        for theta_id in tqdm(tasks, desc=f"Adapting {regime}"):
            full_support = get_task_data(ds_support, theta_id, device)
            support_trajs = full_support[:N_SHOTS]
            query_trajs = get_task_data(ds_query, theta_id, device)
            
            # B. Zero-Shot WITH Refinement
            z_init = infer_z_star(encoder, support_trajs, OBS_LEN)
            obs_data = support_trajs[:, :OBS_LEN, :]
            z_refined = refine_z(sde, z_init, obs_data, cfg, gen, steps=REFINE_STEPS, lr=REFINE_LR)
            
            zs_mse_path, zs_mse_head = evaluate_model(sde, head, z_refined, query_trajs, cfg, gen)
            
            # C. Few-Shot (Using the Refined z) - NOW WITH FIXED HEAD FINE-TUNING
            head_ft = fine_tune_head(sde, head, z_refined, support_trajs, cfg, gen)
            fs_mse_path, fs_mse_head = evaluate_model(sde, head_ft, z_refined, query_trajs, cfg, gen)
            
            # D. From-Scratch (Forecast Mode)
            sde_scr, head_scr, z_scr = train_from_scratch(x_dim, z_dim, support_trajs, cfg, gen)
            scr_mse_path, scr_mse_head = evaluate_model(sde_scr, head_scr, z_scr, query_trajs, cfg, gen)
            
            # Save results
            results_list.append({
                "regime": regime,
                "theta_id": theta_id,
                "n_shots": N_SHOTS,
                
                # Path Errors
                "mse_path_zeroshot": zs_mse_path,
                "mse_path_fewshot": fs_mse_path,
                "mse_path_scratch": scr_mse_path,
                
                # Head Errors (Forecast)
                "mse_head_zeroshot": zs_mse_head,
                "mse_head_fewshot": fs_mse_head,
                "mse_head_scratch": scr_mse_head,
            })
            
    # Save Results
    os.makedirs("results", exist_ok=True)
    df = pd.DataFrame(results_list)
    df.to_csv("results/adaptation_results.csv", index=False)
    
    print("\n✅ Forecasting Adaptation Complete!")
    
    print("\nAverage PATH MSE (Full Trajectory Physics):")
    print(df.groupby("regime")[["mse_path_zeroshot", "mse_path_fewshot", "mse_path_scratch"]].mean())
    
    print("\nAverage FINAL-STEP MSE (Forecasting):")
    print(df.groupby("regime")[["mse_head_zeroshot", "mse_head_fewshot", "mse_head_scratch"]].mean())

if __name__ == "__main__":
    main()


✅ Forecasting Adaptation Complete!

================================================================================
Average PATH MSE (Full Trajectory Physics):
================================================================================
testA    | Zero-Shot: 0.208558 | Few-Shot: 0.208571 | From-Scratch: 0.403132
testB    | Zero-Shot: 0.417241 | Few-Shot: 0.417248 | From-Scratch: 0.626351
testC    | Zero-Shot: 0.695437 | Few-Shot: 0.695482 | From-Scratch: 0.928665

================================================================================
Average FINAL-STEP MSE (Forecasting):
================================================================================
testA    | Zero-Shot: 0.567442 | Few-Shot: 0.270567 | From-Scratch: 0.590878
testB    | Zero-Shot: 1.273147 | Few-Shot: 0.636951 | From-Scratch: 1.886285
testC    | Zero-Shot: 2.281282 | Few-Shot: 0.815658 | From-Scratch: 3.651459

================================================================================
Summary Statistics:
================================================================================
Total tasks evaluated: 90
Regimes: ['testA' 'testB' 'testC']

20 STEPS VERSION THAT SOMEWHAT WORKS

In [ ]:
# adaptation/few_shot_adapt.py
# Meta-Learner Adaptation: Limited "Starvation" View
# We fine-tune using the Prior (z) on just 20 steps.
# Hypothesis: The Prior (z) allows it to extrapolate to 200 steps successfully.

import os
import copy
import pandas as pd
import torch
import torch.optim as optim
from tqdm import tqdm

from config.base_config import cfg
from dataloaders.trajectory_datasets import TrajectoryDataset
from models.encoder import TrajEncoder
from models.neural_sde import NeuralSDE
from models.head import ForecastHead
from training.train_meta import simulate_neural_sde_batch

# --- CONFIG ---
ADAPT_STEPS = 50        # Sufficient for convergence
ADAPT_LR = 1e-2         # High LR for fast adaptation
N_SHOTS = 2
LIMIT_STEPS = 20        # <--- NEW: Limit adaptation to observed past
OBS_LEN_ENCODER = 20    # Context for encoder (should be <= LIMIT_STEPS)

def get_task_data(dataset, theta_id, device):
    rows = dataset.metadata[dataset.metadata["theta_id"] == theta_id]
    indices = rows.index.tolist()
    data = [dataset[i][0] for i in indices]
    return torch.stack(data).to(device)

def infer_z_star(encoder, support_trajs, obs_len):
    """Infer z from the OBSERVED PAST only."""
    encoder.eval()
    with torch.no_grad():
        obs = support_trajs[:, :obs_len, :]
        z_all = encoder(obs)
        z_mean = z_all.mean(dim=0, keepdim=True)
    return z_mean

def evaluate_model(sde, head, z, query_trajs, config, gen):
    """Evaluate on Query set (Full Future T=200)."""
    sde.eval()
    head.eval()
    T = config.time_grid.T
    n_steps = config.time_grid.n_steps
    x_max = config.stability.max_state_abs

    with torch.no_grad():
        x0 = query_trajs[:, 0, :]
        z_expanded = z.expand(x0.size(0), -1)

        traj_pred = simulate_neural_sde_batch(
            sde, x0, z_expanded, T, n_steps, x_max, gen
        )
        x_T_pred = traj_pred[:, -1, :]
        final_pred = head(x_T_pred, z_expanded)

        mse_path = torch.mean((traj_pred - query_trajs) ** 2).item()
        mse_head = torch.mean((final_pred - query_trajs[:, -1, :]) ** 2).item()

    return mse_path, mse_head

def fine_tune_all(encoder, sde, head, support_trajs, config, gen, limit_steps=None):
    """
    Adapt using only the LIMITED horizon (e.g. 50 steps).
    """
    encoder_ft = copy.deepcopy(encoder)
    sde_ft = copy.deepcopy(sde)
    head_ft = copy.deepcopy(head)
    
    params = list(sde_ft.parameters()) + list(head_ft.parameters())
    optimizer = optim.Adam(params, lr=ADAPT_LR)
    
    encoder_ft.eval() 
    sde_ft.train()
    head_ft.train()

    # Calculate Time Horizon for Training
    full_T = config.time_grid.T
    full_steps = config.time_grid.n_steps
    dt = full_T / full_steps

    if limit_steps is not None:
        n_steps_train = limit_steps
        T_train = dt * limit_steps
        # Slice training target (+1 to include t=0)
        train_target = support_trajs[:, :limit_steps+1, :]
    else:
        n_steps_train = full_steps
        T_train = full_T
        train_target = support_trajs

    x_max = config.stability.max_state_abs
    x0 = train_target[:, 0, :]
    target_final_train = train_target[:, -1, :]
    
    with torch.no_grad():
        # Encoder uses OBS_LEN_ENCODER (e.g. 20) which must be <= limit_steps (e.g. 50)
        obs = support_trajs[:, :OBS_LEN_ENCODER, :]
        z = encoder_ft(obs).mean(dim=0, keepdim=True)
        z_expanded = z.expand(x0.size(0), -1)

    for _ in range(ADAPT_STEPS):
        optimizer.zero_grad()
        
        # Simulate ONLY up to limit
        traj_pred = simulate_neural_sde_batch(
            sde_ft, x0, z_expanded, T_train, n_steps_train, x_max, gen
        )
        
        # Predict end of TRAIN slice
        x_T = traj_pred[:, -1, :]
        pred = head_ft(x_T, z_expanded)
        
        loss_path = torch.nn.functional.mse_loss(traj_pred, train_target)
        loss_head = torch.nn.functional.mse_loss(pred, target_final_train)
        
        loss = loss_path + loss_head
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        optimizer.step()

    return encoder_ft, sde_ft, head_ft, z

def main():
    device = torch.device(cfg.device)
    print("🔬 Meta-Learning Adaptation: STARVATION MODE")
    print(f"Config: {N_SHOTS}-Shot | Steps={ADAPT_STEPS} | Limit={LIMIT_STEPS} steps")

    ckpt_path = "checkpoints/meta_epoch_50.pt"
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found at {ckpt_path}")

    ckpt = torch.load(ckpt_path, map_location=device)
    x_dim = cfg.basis.x_dim
    z_dim = cfg.latent.latent_dim

    encoder = TrajEncoder(x_dim, z_dim, cfg.latent.encoder_hidden_dim).to(device)
    sde = NeuralSDE(x_dim, z_dim, cfg.latent.sde_hidden_dim).to(device)
    head = ForecastHead(x_dim, z_dim, cfg.latent.head_hidden_dim).to(device)

    encoder.load_state_dict(ckpt["encoder"])
    sde.load_state_dict(ckpt["sde"])
    head.load_state_dict(ckpt["head"])

    gen = torch.Generator(device=device)
    gen.manual_seed(999)

    results_list = []
    index_path = os.path.join(cfg.paths.data_root, "index.csv")

    for regime in ["testA", "testB", "testC"]:
        print(f"Processing {regime}")
        try:
            ds_support = TrajectoryDataset(index_path, regime, "support", check_shapes=True)
            ds_query = TrajectoryDataset(index_path, regime, "query", check_shapes=True)
        except: continue

        for theta_id in tqdm(ds_support.metadata["theta_id"].unique(), desc=regime):
            full_support = get_task_data(ds_support, theta_id, device)
            
            # A. Zero-shot
            z_star = infer_z_star(encoder, full_support[:N_SHOTS], OBS_LEN_ENCODER)
            zs_mse_path, zs_mse_head = evaluate_model(sde, head, z_star, get_task_data(ds_query, theta_id, device), cfg, gen)

            # B. Few-shot (Restricted to LIMIT_STEPS)
            _, sde_ft, head_ft, z_ft = fine_tune_all(
                encoder, sde, head, full_support[:N_SHOTS], cfg, gen, limit_steps=LIMIT_STEPS
            )
            fs_mse_path, fs_mse_head = evaluate_model(sde_ft, head_ft, z_ft, get_task_data(ds_query, theta_id, device), cfg, gen)

            results_list.append({
                "regime": regime,
                "mse_head_zeroshot": zs_mse_head,
                "mse_head_fewshot": fs_mse_head,
            })

    df = pd.DataFrame(results_list)
    df.to_csv("results/adaptation_results.csv", index=False)
    print("\n✅ Adaptation Complete.")
    print(df.groupby("regime")[["mse_head_zeroshot", "mse_head_fewshot"]].mean())

if __name__ == "__main__":
    main()

20 STEPS
🔬 Meta-Learning Adaptation: STARVATION MODE
Config: 2-Shot | Steps=50 | Limit=20 steps
✅ Adaptation Complete.
        mse_head_zeroshot  mse_head_fewshot
regime                                     
testA            0.390166          0.502157
testB            1.674765          1.627974
testC            3.152233          3.117636

IMROV 1 VERSION

List of changes vs. your pasted version
Introduced explicit z-refinement stage

Added refine_z(sde, z_init, support_past, ...) that:

Uses only support_past (first LIMIT_STEPS points).

Freezes SDE and updates only z.

This replaces direct use of infer_z_star alone in adaptation.

Changed few-shot adaptation from full SDE+head to head-only

Removed fine_tune_all that updated both SDE and head on full/limited trajectories.

Added fine_tune_head_only, which:

Keeps SDE frozen (sde.eval()).

Uses support_past (first LIMIT_STEPS steps).

Fine-tunes only the ForecastHead to match the final observed past point.

Consistent starvation window

Introduced LIMIT_STEPS = 20 and used it consistently for:

support_past = support_trajs[:, :LIMIT_STEPS, :].

Simulation horizon in refine_z and fine_tune_head_only.

Kept OBS_LEN_ENCODER = 20 and used it only for infer_z_star (encoder context).

Separated meta variants conceptually

Meta‑Z (zero-shot): encoder → z_init → refine_z → evaluate with fixed SDE+head.

Meta‑Z+Head (few-shot): same z_refined, then head-only fine‑tune, then evaluate.

Removed scratch baseline from this script

The previous version had train_from_scratch and stored scratch metrics here.

Now this file is strictly meta; scratch lives in baselines/adapt_scratch.py.

Results logging extended

Saving both path and head MSE for zero-shot and few-shot:

mse_path_zeroshot, mse_head_zeroshot, mse_path_fewshot, mse_head_fewshot.

Grouped summaries printed at the end for quick inspection.

This version is ready to run alongside your strong scratch baseline (with its own LIMIT_STEPS = 20) to compare:

Meta‑Z vs scratch.

Meta‑Z+Head vs scratch.

In [ ]:
# adaptation/few_shot_adapt.py
# Meta-Learner Adaptation: z-refinement + Head-only adaptation under STARVATION.
# Encoder + SDE are used as a prior; we refine z on the past and fine-tune only the head.

import os
import copy
import pandas as pd
import torch
import torch.optim as optim
from tqdm import tqdm

from config.base_config import cfg
from dataloaders.trajectory_datasets import TrajectoryDataset
from models.encoder import TrajEncoder
from models.neural_sde import NeuralSDE
from models.head import ForecastHead
from training.train_meta import simulate_neural_sde_batch

# --- CONFIG ---
N_SHOTS = 2

LIMIT_STEPS = 20        # Past window used for adaptation (starvation)
OBS_LEN_ENCODER = 20    # Context for encoder / z refinement (<= LIMIT_STEPS)

REFINE_STEPS = 25       # Steps for z refinement
REFINE_LR = 0.1         # LR for z refinement

ADAPT_STEPS = 50        # Steps for head fine-tuning
ADAPT_LR = 1e-2         # LR for head fine-tuning


def get_task_data(dataset, theta_id, device):
    rows = dataset.metadata[dataset.metadata["theta_id"] == theta_id]
    indices = rows.index.tolist()
    data = [dataset[i][0] for i in indices]
    return torch.stack(data).to(device)


def infer_z_star(encoder, support_trajs, obs_len):
    """Infer initial z from the OBSERVED PAST only (first obs_len steps)."""
    encoder.eval()
    with torch.no_grad():
        obs = support_trajs[:, :obs_len, :]
        z_all = encoder(obs)
        z_mean = z_all.mean(dim=0, keepdim=True)
    return z_mean


def refine_z(sde, z_init, support_past, config, gen, steps=REFINE_STEPS, lr=REFINE_LR):
    """
    Test-time optimization of z on the observed past.
    SDE is frozen; only z is updated to fit the short history.
    """
    z_opt = z_init.clone().detach().requires_grad_(True)
    optimizer = optim.Adam([z_opt], lr=lr)

    T = config.time_grid.T
    n_steps = config.time_grid.n_steps
    dt = T / n_steps

    obs_len = support_past.shape[1]  # LIMIT_STEPS
    T_obs = dt * (obs_len - 1)

    x0 = support_past[:, 0, :]
    x_max = config.stability.max_state_abs

    sde.eval()

    for _ in range(steps):
        optimizer.zero_grad()
        z_expanded = z_opt.expand(x0.size(0), -1)
        traj_pred = simulate_neural_sde_batch(
            sde, x0, z_expanded, T_obs, obs_len - 1, x_max, gen
        )
        loss = torch.nn.functional.mse_loss(traj_pred, support_past)
        loss.backward()
        optimizer.step()

    return z_opt.detach()


def evaluate_model(sde, head, z, query_trajs, config, gen):
    """
    Evaluate on Query set over the FULL horizon [0, T].
    Returns path MSE and final-step MSE.
    """
    sde.eval()
    head.eval()
    T = config.time_grid.T
    n_steps = config.time_grid.n_steps
    x_max = config.stability.max_state_abs

    with torch.no_grad():
        x0 = query_trajs[:, 0, :]
        z_expanded = z.expand(x0.size(0), -1)

        traj_pred = simulate_neural_sde_batch(
            sde, x0, z_expanded, T, n_steps, x_max, gen
        )
        x_T_pred = traj_pred[:, -1, :]
        final_pred = head(x_T_pred, z_expanded)

        mse_path = torch.mean((traj_pred - query_trajs) ** 2).item()
        mse_head = torch.mean((final_pred - query_trajs[:, -1, :]) ** 2).item()

    return mse_path, mse_head


def fine_tune_head_only(sde, head, z, support_past, config, gen,
                        steps=ADAPT_STEPS, lr=ADAPT_LR):
    """
    Few-shot adaptation: fine-tune HEAD ONLY on the observed past (LIMIT_STEPS).
    SDE is frozen and used as a prior; z is fixed (e.g., refined z).
    """
    head_ft = copy.deepcopy(head)
    optimizer = optim.Adam(head_ft.parameters(), lr=lr)

    sde.eval()
    head_ft.train()

    T = config.time_grid.T
    n_steps = config.time_grid.n_steps
    x_max = config.stability.max_state_abs

    obs_len = support_past.shape[1]  # LIMIT_STEPS
    x0 = support_past[:, 0, :]
    target_final_obs = support_past[:, -1, :]

    dt = T / n_steps
    T_obs = dt * (obs_len - 1)

    with torch.no_grad():
        z_expanded = z.expand(x0.size(0), -1)
        traj_pred = simulate_neural_sde_batch(
            sde, x0, z_expanded, T_obs, obs_len - 1, x_max, gen
        )
        x_T_obs = traj_pred[:, -1, :]

    for _ in range(steps):
        optimizer.zero_grad()
        pred = head_ft(x_T_obs, z_expanded)
        loss = torch.nn.functional.mse_loss(pred, target_final_obs)
        loss.backward()
        optimizer.step()

    return head_ft


def main():
    device = torch.device(cfg.device)
    print("🔬 Meta-Learning Adaptation: STARVATION MODE (z-refine + Head-only)")
    print(f"Config: {N_SHOTS}-Shot | LIMIT_STEPS={LIMIT_STEPS} | OBS_LEN_ENCODER={OBS_LEN_ENCODER}")

    ckpt_path = "checkpoints/meta_epoch_50.pt"
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found at {ckpt_path}")

    ckpt = torch.load(ckpt_path, map_location=device)
    x_dim = cfg.basis.x_dim
    z_dim = cfg.latent.latent_dim

    encoder = TrajEncoder(x_dim, z_dim, cfg.latent.encoder_hidden_dim).to(device)
    sde = NeuralSDE(x_dim, z_dim, cfg.latent.sde_hidden_dim).to(device)
    head = ForecastHead(x_dim, z_dim, cfg.latent.head_hidden_dim).to(device)

    encoder.load_state_dict(ckpt["encoder"])
    sde.load_state_dict(ckpt["sde"])
    head.load_state_dict(ckpt["head"])

    gen = torch.Generator(device=device)
    gen.manual_seed(999)

    results_list = []
    index_path = os.path.join(cfg.paths.data_root, "index.csv")

    for regime in ["testA", "testB", "testC"]:
        print(f"Processing {regime}")
        try:
            ds_support = TrajectoryDataset(index_path, regime, "support", check_shapes=True)
            ds_query = TrajectoryDataset(index_path, regime, "query", check_shapes=True)
        except RuntimeError:
            continue

        for theta_id in tqdm(ds_support.metadata["theta_id"].unique(), desc=regime):
            full_support = get_task_data(ds_support, theta_id, device)
            support_trajs = full_support[:N_SHOTS]
            query_trajs = get_task_data(ds_query, theta_id, device)

            # Past slice for encoder + z refinement + head training
            support_past = support_trajs[:, :LIMIT_STEPS, :]

            # A. Zero-shot with z-refinement (Meta-Z)
            z_init = infer_z_star(encoder, support_trajs, OBS_LEN_ENCODER)
            z_refined = refine_z(sde, z_init, support_past, cfg, gen,
                                 steps=REFINE_STEPS, lr=REFINE_LR)
            zs_mse_path, zs_mse_head = evaluate_model(sde, head, z_refined,
                                                      query_trajs, cfg, gen)

            # B. Few-shot: Head-only fine-tuning using refined z (Meta-Z+Head)
            head_ft = fine_tune_head_only(sde, head, z_refined, support_past,
                                          cfg, gen, steps=ADAPT_STEPS, lr=ADAPT_LR)
            fs_mse_path, fs_mse_head = evaluate_model(sde, head_ft, z_refined,
                                                      query_trajs, cfg, gen)

            results_list.append({
                "regime": regime,
                "theta_id": theta_id,
                "mse_path_zeroshot": zs_mse_path,
                "mse_head_zeroshot": zs_mse_head,
                "mse_path_fewshot": fs_mse_path,
                "mse_head_fewshot": fs_mse_head,
            })

    df = pd.DataFrame(results_list)
    os.makedirs("results", exist_ok=True)
    df.to_csv("results/adaptation_results.csv", index=False)
    print("\n✅ Adaptation Complete.")
    print("\n📈 Average PATH MSE:")
    print(df.groupby("regime")[["mse_path_zeroshot", "mse_path_fewshot"]].mean())
    print("\n📈 Average FINAL-STEP MSE:")
    print(df.groupby("regime")[["mse_head_zeroshot", "mse_head_fewshot"]].mean())


if __name__ == "__main__":
    main()


✅ Adaptation Complete.

📈 Average PATH MSE:
        mse_path_zeroshot  mse_path_fewshot
regime                                     
testA            0.208644          0.208651
testB            0.417805          0.417813
testC            0.696276          0.696329

📈 Average FINAL-STEP MSE:
        mse_head_zeroshot  mse_head_fewshot
regime                                     
testA            0.567522          0.452897
testB            1.277736          1.698503
testC            2.283824          3.034561

IMPROV 2 VERSION 

In [ ]:

# Meta-Learner Adaptation: z-refinement + Head-only adaptation under STARVATION.
# Encoder + SDE are used as a prior; we refine z on the past and fine-tune only the head.

import os
import copy
import pandas as pd
import torch
import torch.optim as optim
from tqdm import tqdm

from config.base_config import cfg
from dataloaders.trajectory_datasets import TrajectoryDataset
from models.encoder import TrajEncoder
from models.neural_sde import NeuralSDE
from models.head import ForecastHead
from training.train_meta import simulate_neural_sde_batch

# --- CONFIG ---
N_SHOTS = 2

LIMIT_STEPS = 20        # Past window used for adaptation (starvation)
OBS_LEN_ENCODER = 20    # Context for encoder / z refinement (<= LIMIT_STEPS)

REFINE_STEPS = 25       # Steps for z refinement
REFINE_LR = 0.1         # LR for z refinement

ADAPT_STEPS = 20        # Steps for head fine-tuning
ADAPT_LR = 1e-3         # LR for head fine-tuning


def get_task_data(dataset, theta_id, device):
    rows = dataset.metadata[dataset.metadata["theta_id"] == theta_id]
    indices = rows.index.tolist()
    data = [dataset[i][0] for i in indices]
    return torch.stack(data).to(device)


def infer_z_star(encoder, support_trajs, obs_len):
    """Infer initial z from the OBSERVED PAST only (first obs_len steps)."""
    encoder.eval()
    with torch.no_grad():
        obs = support_trajs[:, :obs_len, :]
        z_all = encoder(obs)
        z_mean = z_all.mean(dim=0, keepdim=True)
    return z_mean


def refine_z(sde, z_init, support_past, config, gen, steps=REFINE_STEPS, lr=REFINE_LR):
    """
    Test-time optimization of z on the observed past.
    SDE is frozen; only z is updated to fit the short history.
    """
    z_opt = z_init.clone().detach().requires_grad_(True)
    optimizer = optim.Adam([z_opt], lr=lr)

    T = config.time_grid.T
    n_steps = config.time_grid.n_steps
    dt = T / n_steps

    obs_len = support_past.shape[1]  # LIMIT_STEPS
    T_obs = dt * (obs_len - 1)

    x0 = support_past[:, 0, :]
    x_max = config.stability.max_state_abs

    sde.eval()

    for _ in range(steps):
        optimizer.zero_grad()
        z_expanded = z_opt.expand(x0.size(0), -1)
        traj_pred = simulate_neural_sde_batch(
            sde, x0, z_expanded, T_obs, obs_len - 1, x_max, gen
        )
        loss = torch.nn.functional.mse_loss(traj_pred, support_past)
        loss.backward()
        optimizer.step()

    return z_opt.detach()


def evaluate_model(sde, head, z, query_trajs, config, gen):
    """
    Evaluate on Query set over the FULL horizon [0, T].
    Returns path MSE and final-step MSE.
    """
    sde.eval()
    head.eval()
    T = config.time_grid.T
    n_steps = config.time_grid.n_steps
    x_max = config.stability.max_state_abs

    with torch.no_grad():
        x0 = query_trajs[:, 0, :]
        z_expanded = z.expand(x0.size(0), -1)

        traj_pred = simulate_neural_sde_batch(
            sde, x0, z_expanded, T, n_steps, x_max, gen
        )
        x_T_pred = traj_pred[:, -1, :]
        final_pred = head(x_T_pred, z_expanded)

        mse_path = torch.mean((traj_pred - query_trajs) ** 2).item()
        mse_head = torch.mean((final_pred - query_trajs[:, -1, :]) ** 2).item()

    return mse_path, mse_head


def fine_tune_head_only(sde, head, z, support_past, config, gen,
                        steps=ADAPT_STEPS, lr=ADAPT_LR):
    """
    Few-shot adaptation: fine-tune HEAD ONLY on the observed past (LIMIT_STEPS).
    SDE is frozen and used as a prior; z is fixed (e.g., refined z).
    """
    head_ft = copy.deepcopy(head)
    optimizer = optim.Adam(head_ft.parameters(), lr=lr)

    sde.eval()
    head_ft.train()

    T = config.time_grid.T
    n_steps = config.time_grid.n_steps
    x_max = config.stability.max_state_abs

    obs_len = support_past.shape[1]  # LIMIT_STEPS
    x0 = support_past[:, 0, :]
    target_final_obs = support_past[:, -1, :]

    dt = T / n_steps
    T_obs = dt * (obs_len - 1)

    with torch.no_grad():
        z_expanded = z.expand(x0.size(0), -1)
        traj_pred = simulate_neural_sde_batch(
            sde, x0, z_expanded, T_obs, obs_len - 1, x_max, gen
        )
        x_T_obs = traj_pred[:, -1, :]

    for _ in range(steps):
        optimizer.zero_grad()
        pred = head_ft(x_T_obs, z_expanded)
        loss = torch.nn.functional.mse_loss(pred, target_final_obs)
        loss.backward()
        optimizer.step()

    return head_ft


def main():
    device = torch.device(cfg.device)
    print("🔬 Meta-Learning Adaptation: STARVATION MODE (z-refine + Head-only)")
    print(f"Config: {N_SHOTS}-Shot | LIMIT_STEPS={LIMIT_STEPS} | OBS_LEN_ENCODER={OBS_LEN_ENCODER}")

    ckpt_path = "checkpoints/meta_epoch_50.pt"
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found at {ckpt_path}")

    ckpt = torch.load(ckpt_path, map_location=device)
    x_dim = cfg.basis.x_dim
    z_dim = cfg.latent.latent_dim

    encoder = TrajEncoder(x_dim, z_dim, cfg.latent.encoder_hidden_dim).to(device)
    sde = NeuralSDE(x_dim, z_dim, cfg.latent.sde_hidden_dim).to(device)
    head = ForecastHead(x_dim, z_dim, cfg.latent.head_hidden_dim).to(device)

    encoder.load_state_dict(ckpt["encoder"])
    sde.load_state_dict(ckpt["sde"])
    head.load_state_dict(ckpt["head"])

    gen = torch.Generator(device=device)
    gen.manual_seed(999)

    results_list = []
    index_path = os.path.join(cfg.paths.data_root, "index.csv")

    for regime in ["testA", "testB", "testC"]:
        print(f"Processing {regime}")
        try:
            ds_support = TrajectoryDataset(index_path, regime, "support", check_shapes=True)
            ds_query = TrajectoryDataset(index_path, regime, "query", check_shapes=True)
        except RuntimeError:
            continue

        for theta_id in tqdm(ds_support.metadata["theta_id"].unique(), desc=regime):
            full_support = get_task_data(ds_support, theta_id, device)
            support_trajs = full_support[:N_SHOTS]
            query_trajs = get_task_data(ds_query, theta_id, device)

            # Past slice for encoder + z refinement + head training
            support_past = support_trajs[:, :LIMIT_STEPS, :]

            # A. Zero-shot with z-refinement (Meta-Z)
            z_init = infer_z_star(encoder, support_trajs, OBS_LEN_ENCODER)
            z_refined = refine_z(sde, z_init, support_past, cfg, gen,
                                 steps=REFINE_STEPS, lr=REFINE_LR)
            zs_mse_path, zs_mse_head = evaluate_model(sde, head, z_refined,
                                                      query_trajs, cfg, gen)

            # B. Few-shot: Head-only fine-tuning using refined z (Meta-Z+Head)
            head_ft = fine_tune_head_only(sde, head, z_refined, support_past,
                                          cfg, gen, steps=ADAPT_STEPS, lr=ADAPT_LR)
            fs_mse_path, fs_mse_head = evaluate_model(sde, head_ft, z_refined,
                                                      query_trajs, cfg, gen)

            results_list.append({
                "regime": regime,
                "theta_id": theta_id,
                "mse_path_zeroshot": zs_mse_path,
                "mse_head_zeroshot": zs_mse_head,
                "mse_path_fewshot": fs_mse_path,
                "mse_head_fewshot": fs_mse_head,
            })

    df = pd.DataFrame(results_list)
    os.makedirs("results", exist_ok=True)
    df.to_csv("results/adaptation_results.csv", index=False)
    print("\n✅ Adaptation Complete.")
    print("\n📈 Average PATH MSE:")
    print(df.groupby("regime")[["mse_path_zeroshot", "mse_path_fewshot"]].mean())
    print("\n📈 Average FINAL-STEP MSE:")
    print(df.groupby("regime")[["mse_head_zeroshot", "mse_head_fewshot"]].mean())


if __name__ == "__main__":
    main()


✅ Adaptation Complete.

📈 Average PATH MSE:
        mse_path_zeroshot  mse_path_fewshot
regime                                     
testA            0.208644          0.208651
testB            0.417805          0.417813
testC            0.696276          0.696329

📈 Average FINAL-STEP MSE:
        mse_head_zeroshot  mse_head_fewshot
regime                                     
testA            0.567522          0.464740
testB            1.277736          1.357610
testC            2.283824          2.354252
jovyan@jupyter-ec241027:~/DISSERTATION $ 

like improv 2 ABOVE HERE BUT FULL TRAJECTORIES

In [ ]:
# adaptation/few_shot_adapt.py
# HERO RUN: Hybrid Adaptation (Freeze -> Thaw) on Full Trajectory (System ID)
# This combines data availability (System ID) with stability (Hybrid Training).

import os
import copy
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

from config.base_config import cfg
from dataloaders.trajectory_datasets import TrajectoryDataset
from models.encoder import TrajEncoder
from models.neural_sde import NeuralSDE
from models.head import ForecastHead
from training.train_meta import simulate_neural_sde_batch

# --- CONFIG ---
N_SHOTS = 2

# HYBRID SCHEDULE
ADAPT_STEPS_PHASE1 = 50     # Align Head/Encoder (SDE Frozen)
ADAPT_STEPS_PHASE2 = 50     # Fine-tune SDE (SDE Unfrozen, Low LR)

LR_HEAD = 1e-2              # Fast adaptation for Readout
LR_SDE = 1e-4               # Slow, careful adaptation for Physics

# DATA ACCESS
# OBS_LEN_ENCODER = 20      # Context for Encoder (The "Prompt")
# LIMIT_STEPS = None        # NONE = Use Full Trajectory for Training (System ID)

def get_task_data(dataset, theta_id, device):
    rows = dataset.metadata[dataset.metadata["theta_id"] == theta_id]
    indices = rows.index.tolist()
    data = [dataset[i][0] for i in indices]
    return torch.stack(data).to(device)

def infer_z_star(encoder, support_trajs, obs_len):
    """Infer z from the OBSERVED PAST only."""
    encoder.eval()
    with torch.no_grad():
        obs = support_trajs[:, :obs_len, :]
        z_all = encoder(obs)
        z_mean = z_all.mean(dim=0, keepdim=True)
    return z_mean

def evaluate_model(sde, head, z, query_trajs, config, gen):
    """Evaluate on Query set (Full Future)."""
    sde.eval()
    head.eval()
    T = config.time_grid.T
    n_steps = config.time_grid.n_steps
    x_max = config.stability.max_state_abs

    with torch.no_grad():
        x0 = query_trajs[:, 0, :]
        z_expanded = z.expand(x0.size(0), -1)

        traj_pred = simulate_neural_sde_batch(
            sde, x0, z_expanded, T, n_steps, x_max, gen
        )
        x_T_pred = traj_pred[:, -1, :]
        final_pred = head(x_T_pred, z_expanded)

        mse_path = torch.mean((traj_pred - query_trajs) ** 2).item()
        mse_head = torch.mean((final_pred - query_trajs[:, -1, :]) ** 2).item()

    return mse_path, mse_head

def fine_tune_hybrid(encoder, sde, head, support_trajs, config, gen):
    """
    Hybrid Adaptation on FULL Trajectory.
    Phase 1: Freeze SDE, align Head/Encoder.
    Phase 2: Unfreeze SDE (Low LR) for fine-grained physics correction.
    """
    encoder_ft = copy.deepcopy(encoder)
    sde_ft = copy.deepcopy(sde)
    head_ft = copy.deepcopy(head)
    
    # Setup Data (Full Trajectory)
    full_target = support_trajs
    x0 = full_target[:, 0, :]
    target_final = full_target[:, -1, :]
    
    T = config.time_grid.T
    n_steps = config.time_grid.n_steps
    x_max = config.stability.max_state_abs
    
    # ---------------------------------------------------------
    # PHASE 1: ALIGNMENT (Freeze SDE)
    # ---------------------------------------------------------
    # Only train Head and Encoder. SDE is anchor.
    
    param_group_1 = list(encoder_ft.parameters()) + list(head_ft.parameters())
    optimizer_1 = optim.Adam(param_group_1, lr=LR_HEAD)
    
    sde_ft.eval()     
    encoder_ft.train()
    head_ft.train()
    
    # Explicitly freeze SDE
    for p in sde_ft.parameters():
        p.requires_grad = False

    # Note: Encoder only sees first 20 steps to get z, but we calculate loss on FULL path
    obs_len = 20 
    
    for _ in range(ADAPT_STEPS_PHASE1):
        optimizer_1.zero_grad()
        
        # 1. Infer z from past
        obs = full_target[:, :obs_len, :]
        z = encoder_ft(obs).mean(dim=0, keepdim=True)
        z_expanded = z.expand(x0.size(0), -1)
        
        # 2. Simulate Full Path (SDE is frozen)
        traj_pred = simulate_neural_sde_batch(
            sde_ft, x0, z_expanded, T, n_steps, x_max, gen
        )
        
        # 3. Loss on Full Path + Final Step
        x_T = traj_pred[:, -1, :]
        pred = head_ft(x_T, z_expanded)
        
        loss = nn.functional.mse_loss(traj_pred, full_target) + \
               nn.functional.mse_loss(pred, target_final)
        
        loss.backward()
        optimizer_1.step()

    # ---------------------------------------------------------
    # PHASE 2: FINE-TUNING (Unfreeze SDE, Low LR)
    # ---------------------------------------------------------
    # Allow tiny physics updates
    
    for p in sde_ft.parameters():
        p.requires_grad = True
        
    sde_ft.train()
    
    optimizer_2 = optim.Adam([
        {'params': encoder_ft.parameters(), 'lr': LR_HEAD * 0.1}, 
        {'params': head_ft.parameters(),    'lr': LR_HEAD * 0.1}, 
        {'params': sde_ft.parameters(),     'lr': LR_SDE}         # 1e-4 for SDE
    ])
    
    for _ in range(ADAPT_STEPS_PHASE2):
        optimizer_2.zero_grad()
        
        obs = full_target[:, :obs_len, :]
        z = encoder_ft(obs).mean(dim=0, keepdim=True)
        z_expanded = z.expand(x0.size(0), -1)
        
        traj_pred = simulate_neural_sde_batch(
            sde_ft, x0, z_expanded, T, n_steps, x_max, gen
        )
        
        x_T = traj_pred[:, -1, :]
        pred = head_ft(x_T, z_expanded)
        
        loss = nn.functional.mse_loss(traj_pred, full_target) + \
               nn.functional.mse_loss(pred, target_final)
        
        loss.backward()
        nn.utils.clip_grad_norm_(sde_ft.parameters(), 1.0) 
        optimizer_2.step()
        
    return sde_ft, head_ft, z

def main():
    device = torch.device(cfg.device)
    print("🔬 Meta-Learning: HYBRID ADAPTATION + SYSTEM ID (Full Trajectory)")
    print(f"Phase 1: {ADAPT_STEPS_PHASE1} steps (SDE Frozen)")
    print(f"Phase 2: {ADAPT_STEPS_PHASE2} steps (SDE Unfrozen, LR={LR_SDE})")

    ckpt_path = "checkpoints/meta_epoch_50.pt"
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found at {ckpt_path}")

    ckpt = torch.load(ckpt_path, map_location=device)
    x_dim = cfg.basis.x_dim
    z_dim = cfg.latent.latent_dim

    encoder = TrajEncoder(x_dim, z_dim, cfg.latent.encoder_hidden_dim).to(device)
    sde = NeuralSDE(x_dim, z_dim, cfg.latent.sde_hidden_dim).to(device)
    head = ForecastHead(x_dim, z_dim, cfg.latent.head_hidden_dim).to(device)

    encoder.load_state_dict(ckpt["encoder"])
    sde.load_state_dict(ckpt["sde"])
    head.load_state_dict(ckpt["head"])

    gen = torch.Generator(device=device)
    gen.manual_seed(999)

    results_list = []
    index_path = os.path.join(cfg.paths.data_root, "index.csv")

    for regime in ["testA", "testB", "testC"]:
        print(f"Processing {regime}")
        try:
            ds_support = TrajectoryDataset(index_path, regime, "support", check_shapes=True)
            ds_query = TrajectoryDataset(index_path, regime, "query", check_shapes=True)
        except: continue

        for theta_id in tqdm(ds_support.metadata["theta_id"].unique(), desc=regime):
            full_support = get_task_data(ds_support, theta_id, device)
            
            # A. Zero-shot
            z_star = infer_z_star(encoder, full_support[:N_SHOTS], 20)
            zs_mse_path, zs_mse_head = evaluate_model(sde, head, z_star, get_task_data(ds_query, theta_id, device), cfg, gen)

            # B. Hybrid Few-shot (Full Traj)
            sde_ft, head_ft, z_ft = fine_tune_hybrid(
                encoder, sde, head, full_support[:N_SHOTS], cfg, gen
            )
            fs_mse_path, fs_mse_head = evaluate_model(sde_ft, head_ft, z_ft, get_task_data(ds_query, theta_id, device), cfg, gen)

            results_list.append({
                "regime": regime,
                "mse_head_zeroshot": zs_mse_head,
                "mse_head_fewshot": fs_mse_head,
            })

    df = pd.DataFrame(results_list)
    df.to_csv("results/adaptation_results.csv", index=False)
    print("\n✅ Hybrid Adaptation Complete.")
    print(df.groupby("regime")[["mse_head_zeroshot", "mse_head_fewshot"]].mean())

if __name__ == "__main__":
    main()

🔬 Meta-Learning: HYBRID ADAPTATION + SYSTEM ID (Full Trajectory)
Phase 1: 50 steps (SDE Frozen)
Phase 2: 50 steps (SDE Unfrozen, LR=0.0001)
✅ Hybrid Adaptation Complete.
        mse_head_zeroshot  mse_head_fewshot
regime                                     
testA            0.390171          0.274213
testB            1.674748          0.612894
testC            3.152179          0.784839

SAME AS ABOVE (improv 2 on full trajectories) BUT AFTER UPDATING TRAINING . adptation code same as cell above, below find new training code 

In [ ]:
# training/train_meta.py
# Meta-Training Loop with Randomized Context (Robustness Fix)

import os
import torch
import torch.nn.functional as F
from torch import optim
from torch.utils.data import DataLoader
from tqdm import tqdm

from config.base_config import cfg
from dataloaders.trajectory_datasets import TrajectoryDataset
from models.encoder import TrajEncoder
from models.neural_sde import NeuralSDE
from models.head import ForecastHead

# -----------------------------
# Hyperparameters
# -----------------------------
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
N_EPOCHS = 50
OBS_LEN = 50       # Max context training sees
LAMBDA_HEAD = 0.1  # Weight for forecasting loss

# -----------------------------
# Simulator Function
# -----------------------------
def simulate_neural_sde_batch(
    sde: NeuralSDE,
    x0: torch.Tensor,         # (batch, d)
    z: torch.Tensor,          # (batch, z_dim)
    T: float,
    n_steps: int,
    x_max_abs: float,
    generator: torch.Generator,
) -> torch.Tensor:
    """
    Euler–Maruyama simulation using the learned NeuralSDE.
    Returns: traj: (batch, n_steps + 1, d)
    """
    device = x0.device
    batch_size, d = x0.shape
    dt = T / n_steps
    sqrt_dt = dt ** 0.5
    
    traj = torch.zeros(batch_size, n_steps + 1, d, device=device)
    x = x0.clone()
    traj[:, 0, :] = x
    
    for k in range(n_steps):
        t = torch.tensor(k * dt, device=device)
        
        # Drift and diffusion
        b = sde.f(t, x, z)              # (batch, d)
        G = sde.g(t, x, z)              # (batch, d, d) diagonal
        
        # Brownian increment
        dW = torch.randn(batch_size, d, device=device, generator=generator) * sqrt_dt
        
        # Noise term: G @ dW
        noise = torch.bmm(G, dW.unsqueeze(-1)).squeeze(-1)
        
        # EM step
        x = x + b * dt + noise
        
        # Clamp state for numerical stability
        x = torch.clamp(x, -x_max_abs, x_max_abs)
        
        traj[:, k + 1, :] = x
        
    return traj


# -----------------------------
# Training Loop
# -----------------------------
def train_meta_loop():
    device = torch.device(cfg.device)
    print(f"🚀 Starting meta-training on {device}")
    
    index_path = os.path.join(cfg.paths.data_root, "index.csv")
    if not os.path.exists(index_path):
        raise FileNotFoundError(f"index.csv not found at {index_path}.")

    # -----------------
    # Datasets
    # -----------------
    train_ds = TrajectoryDataset(index_path, "train", "train_inner", check_shapes=True)
    val_ds = TrajectoryDataset(index_path, "val", "val", check_shapes=True)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

    print(f"Train trajectories: {len(train_ds)} | Val trajectories: {len(val_ds)}")

    # -----------------
    # Models
    # -----------------
    x_dim = cfg.basis.x_dim
    z_dim = cfg.latent.latent_dim
    
    # Ensure you increased encoder_hidden_dim in config/base_config.py!
    encoder = TrajEncoder(x_dim, z_dim, cfg.latent.encoder_hidden_dim, num_layers=2, dropout=0.1).to(device)
    sde = NeuralSDE(x_dim, z_dim, cfg.latent.sde_hidden_dim).to(device)
    head = ForecastHead(x_dim, z_dim, cfg.latent.head_hidden_dim).to(device)

    params = list(encoder.parameters()) + list(sde.parameters()) + list(head.parameters())
    optimizer = optim.Adam(params, lr=LEARNING_RATE)
    
    gen = torch.Generator(device=device)
    gen.manual_seed(cfg.global_seed + 4242)

    T = cfg.time_grid.T
    n_steps = cfg.time_grid.n_steps
    x_max_abs = cfg.stability.max_state_abs

    os.makedirs("checkpoints", exist_ok=True)

    # -----------------
    # Training Loop
    # -----------------
    for epoch in range(1, N_EPOCHS + 1):
        encoder.train()
        sde.train()
        head.train()
        
        running_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{N_EPOCHS}")

        for traj_batch, _ in pbar:
            traj_batch = traj_batch.to(device)
            B, T_total, d_ = traj_batch.shape

            # -------- A. Encoder: Randomized Context (Robustness) --------
            # FIX: Randomly sample length between 20 and 50.
            # This teaches the encoder to handle "starvation" (short inputs).
            current_obs_len = torch.randint(low=20, high=OBS_LEN + 1, size=(1,)).item()
            
            obs = traj_batch[:, :current_obs_len, :] 
            z = encoder(obs)

            # -------- B. Neural SDE: Simulate Full Path --------
            x0 = traj_batch[:, 0, :]
            traj_pred = simulate_neural_sde_batch(
                sde, x0, z, T, n_steps, x_max_abs, gen
            )

            # Align lengths if needed
            if traj_pred.shape[1] != T_total:
                min_T = min(traj_pred.shape[1], T_total)
                traj_pred = traj_pred[:, :min_T, :]
                traj_true = traj_batch[:, :min_T, :]
            else:
                traj_true = traj_batch

            # -------- C. Forecast Head --------
            final_pred_sde = traj_pred[:, -1, :]
            head_pred = head(final_pred_sde, z)
            target_final = traj_true[:, -1, :]

            # -------- D. Loss --------
            loss_path = F.mse_loss(traj_pred, traj_true)
            loss_head = F.mse_loss(head_pred, target_final)
            loss = loss_path + LAMBDA_HEAD * loss_head

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, max_norm=1.0)
            optimizer.step()

            running_loss += loss.item()
            pbar.set_postfix({"loss": loss.item()})

        # -----------------
        # Validation & Save
        # -----------------
        if epoch % 10 == 0:
            ckpt_path = os.path.join("checkpoints", f"meta_epoch_{epoch}.pt")
            torch.save({
                "encoder": encoder.state_dict(),
                "sde": sde.state_dict(),
                "head": head.state_dict(),
                "cfg": cfg,
            }, ckpt_path)
            print(f"💾 Saved checkpoint: {ckpt_path}")

if __name__ == "__main__":
    train_meta_loop()

✅ Hybrid Adaptation Complete.
        mse_head_zeroshot  mse_head_fewshot
regime                                     
testA            0.390551          0.277866
testB            1.629351          0.613605
testC            3.007632          0.803846

In [ ]:
LIKE BEFORE BUT WITH MANY DIFFERENT STEPS LENGTH 

In [ ]:
# adaptation/few_shot_adapt.py
# HERO RUN: Data Efficiency Sweep (Granular)
# Measures performance as a function of available support steps.

import os
import copy
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

from config.base_config import cfg
from dataloaders.trajectory_datasets import TrajectoryDataset
from models.encoder import TrajEncoder
from models.neural_sde import NeuralSDE
from models.head import ForecastHead
from training.train_meta import simulate_neural_sde_batch

# --- CONFIG ---
N_SHOTS = 2

# Hybrid Schedule
ADAPT_STEPS_PHASE1 = 50     # Freeze SDE
ADAPT_STEPS_PHASE2 = 50     # Unfreeze SDE (Low LR)
LR_HEAD = 1e-2              
LR_SDE = 1e-4

# THE GRANULAR SWEEP
# 20  = Starvation (Fail zone)
# 40  = Very Sparse
# 50  = Sparse
# 80  = Mid-range
# 100 = Half Trajectory
# 120 = Majority Trajectory
# 201 = Full Trajectory (System ID)
STEPS_SWEEP = [20, 40, 50, 80, 100, 120, 201]

def get_task_data(dataset, theta_id, device):
    rows = dataset.metadata[dataset.metadata["theta_id"] == theta_id]
    indices = rows.index.tolist()
    data = [dataset[i][0] for i in indices]
    return torch.stack(data).to(device)

def infer_z_star(encoder, support_trajs, obs_len):
    """Infer z from the OBSERVED PAST only."""
    encoder.eval()
    with torch.no_grad():
        obs = support_trajs[:, :obs_len, :]
        z_all = encoder(obs)
        z_mean = z_all.mean(dim=0, keepdim=True)
    return z_mean

def evaluate_model(sde, head, z, query_trajs, config, gen):
    """Evaluate on Query set (Full Future 0->T)."""
    sde.eval()
    head.eval()
    T = config.time_grid.T
    n_steps = config.time_grid.n_steps
    x_max = config.stability.max_state_abs

    with torch.no_grad():
        x0 = query_trajs[:, 0, :]
        z_expanded = z.expand(x0.size(0), -1)

        # Always simulate full duration for evaluation
        traj_pred = simulate_neural_sde_batch(
            sde, x0, z_expanded, T, n_steps, x_max, gen
        )
        x_T_pred = traj_pred[:, -1, :]
        final_pred = head(x_T_pred, z_expanded)

        mse_path = torch.mean((traj_pred - query_trajs) ** 2).item()
        mse_head = torch.mean((final_pred - query_trajs[:, -1, :]) ** 2).item()

    return mse_path, mse_head

def fine_tune_hybrid(encoder, sde, head, support_trajs, config, gen, limit_steps):
    """
    Hybrid Adaptation on LIMITED History.
    1. Slices support_trajs to [:limit_steps].
    2. Simulates only up to corresponding T_train.
    """
    encoder_ft = copy.deepcopy(encoder)
    sde_ft = copy.deepcopy(sde)
    head_ft = copy.deepcopy(head)
    
    # 1. Prepare Training Data (Slice based on availability)
    train_target = support_trajs[:, :limit_steps, :]
    x0 = train_target[:, 0, :]
    target_final = train_target[:, -1, :]
    
    # 2. Calculate Simulation Params for this partial horizon
    T_full = config.time_grid.T
    n_steps_full = config.time_grid.n_steps
    dt = T_full / n_steps_full
    
    # number of simulation steps = points - 1 (since t=0 is start)
    # limit_steps is e.g. 20 (indices 0..19). We need 19 steps to get there.
    n_sim = limit_steps - 1
    T_train = dt * n_sim
    x_max = config.stability.max_state_abs
    
    # Encoder sees up to 50 steps max, or limit_steps if smaller
    enc_obs_len = min(limit_steps, 50)

    # ---------------------------------------------------------
    # PHASE 1: ALIGNMENT (Freeze SDE)
    # ---------------------------------------------------------
    param_group_1 = list(encoder_ft.parameters()) + list(head_ft.parameters())
    optimizer_1 = optim.Adam(param_group_1, lr=LR_HEAD)
    
    sde_ft.eval()     
    encoder_ft.train()
    head_ft.train()
    
    for p in sde_ft.parameters():
        p.requires_grad = False

    for _ in range(ADAPT_STEPS_PHASE1):
        optimizer_1.zero_grad()
        
        # Infer z
        obs = train_target[:, :enc_obs_len, :]
        z = encoder_ft(obs).mean(dim=0, keepdim=True)
        z_expanded = z.expand(x0.size(0), -1)
        
        # Simulate Partial Path
        traj_pred = simulate_neural_sde_batch(
            sde_ft, x0, z_expanded, T_train, n_sim, x_max, gen
        )
        
        x_T = traj_pred[:, -1, :]
        pred = head_ft(x_T, z_expanded)
        
        loss = nn.functional.mse_loss(traj_pred, train_target) + \
               nn.functional.mse_loss(pred, target_final)
        
        loss.backward()
        optimizer_1.step()

    # ---------------------------------------------------------
    # PHASE 2: FINE-TUNING (Unfreeze SDE, Low LR)
    # ---------------------------------------------------------
    for p in sde_ft.parameters():
        p.requires_grad = True
        
    sde_ft.train()
    
    optimizer_2 = optim.Adam([
        {'params': encoder_ft.parameters(), 'lr': LR_HEAD * 0.1}, 
        {'params': head_ft.parameters(),    'lr': LR_HEAD * 0.1}, 
        {'params': sde_ft.parameters(),     'lr': LR_SDE}
    ])
    
    for _ in range(ADAPT_STEPS_PHASE2):
        optimizer_2.zero_grad()
        
        obs = train_target[:, :enc_obs_len, :]
        z = encoder_ft(obs).mean(dim=0, keepdim=True)
        z_expanded = z.expand(x0.size(0), -1)
        
        traj_pred = simulate_neural_sde_batch(
            sde_ft, x0, z_expanded, T_train, n_sim, x_max, gen
        )
        
        x_T = traj_pred[:, -1, :]
        pred = head_ft(x_T, z_expanded)
        
        loss = nn.functional.mse_loss(traj_pred, train_target) + \
               nn.functional.mse_loss(pred, target_final)
        
        loss.backward()
        nn.utils.clip_grad_norm_(sde_ft.parameters(), 1.0) 
        optimizer_2.step()
        
    return sde_ft, head_ft, z

def main():
    device = torch.device(cfg.device)
    print("🔬 Meta-Learning: Data Efficiency Sweep")
    print(f"Sweeping Step Limits: {STEPS_SWEEP}")
    print(f"Hybrid Schedule: {ADAPT_STEPS_PHASE1} (Freeze) + {ADAPT_STEPS_PHASE2} (Thaw)")

    ckpt_path = "checkpoints/meta_epoch_50.pt"
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found at {ckpt_path}. Run training first.")

    ckpt = torch.load(ckpt_path, map_location=device)
    x_dim = cfg.basis.x_dim
    z_dim = cfg.latent.latent_dim

    # Load Base Models
    encoder_base = TrajEncoder(x_dim, z_dim, cfg.latent.encoder_hidden_dim).to(device)
    sde_base = NeuralSDE(x_dim, z_dim, cfg.latent.sde_hidden_dim).to(device)
    head_base = ForecastHead(x_dim, z_dim, cfg.latent.head_hidden_dim).to(device)

    encoder_base.load_state_dict(ckpt["encoder"])
    sde_base.load_state_dict(ckpt["sde"])
    head_base.load_state_dict(ckpt["head"])

    gen = torch.Generator(device=device)
    gen.manual_seed(999)

    results_list = []
    index_path = os.path.join(cfg.paths.data_root, "index.csv")

    # Regimes Loop
    for regime in ["testA", "testB", "testC"]:
        print(f"\n🎯 Processing {regime}")
        try:
            ds_support = TrajectoryDataset(index_path, regime, "support", check_shapes=True)
            ds_query = TrajectoryDataset(index_path, regime, "query", check_shapes=True)
        except: continue

        # Tasks Loop
        tasks = ds_support.metadata["theta_id"].unique()
        for theta_id in tqdm(tasks, desc=regime):
            full_support = get_task_data(ds_support, theta_id, device)
            full_query = get_task_data(ds_query, theta_id, device)
            
            # --- THE SWEEP ---
            for limit_steps in STEPS_SWEEP:
                # 1. Zero-shot (Uses only limit_steps for context)
                enc_obs_len = min(limit_steps, 50)
                z_star = infer_z_star(encoder_base, full_support[:N_SHOTS], enc_obs_len)
                zs_mse_path, zs_mse_head = evaluate_model(
                    sde_base, head_base, z_star, full_query, cfg, gen
                )

                # 2. Hybrid Few-shot (Trains on limit_steps)
                sde_ft, head_ft, z_ft = fine_tune_hybrid(
                    encoder_base, sde_base, head_base, 
                    full_support[:N_SHOTS], cfg, gen, limit_steps
                )
                
                fs_mse_path, fs_mse_head = evaluate_model(
                    sde_ft, head_ft, z_ft, full_query, cfg, gen
                )

                results_list.append({
                    "regime": regime,
                    "theta_id": theta_id,
                    "steps_available": limit_steps,
                    "mse_head_zeroshot": zs_mse_head,
                    "mse_head_fewshot": fs_mse_head,
                    "mse_path_fewshot": fs_mse_path
                })

    # Save Results
    df = pd.DataFrame(results_list)
    out_path = "results/efficiency_sweep_results.csv"
    os.makedirs("results", exist_ok=True)
    df.to_csv(out_path, index=False)
    
    print("\n✅ Sweep Complete.")
    print("\n📊 Average Head MSE by Regime & Steps:")
    print(df.groupby(["regime", "steps_available"])[["mse_head_fewshot"]].mean())

if __name__ == "__main__":
    main()

mse_head_fewshot
regime steps_available
testA 
20 0.415778
40 0.397880
50 0.392155
80 0.376083
100 0.336949
120 0.314229
201 0.280373
testB 
20 1.718360
40 1.548125
50 1.453675
80 1.143014
100 0.966035
120 0.839822
201 0.617184
testC 
20 3.315244
40 2.883624
50 2.725842
80 2.037143
100 1.545862
120 1.247378
201 0.796292

WHAT CHANGED : 
Loss during adaptation:

Before: MSE(traj_pred, train_target) + MSE(pred, target_final) (path + final).

Now: MSE(pred, target_final) only.

This aligns the adaptation objective with your evaluation metric (final-step MSE) and avoids noisy supervision at every time step when only short prefixes are available.
​

No SDE fine-tuning in few-shot:

Before: Phase 1 (encoder+head) + Phase 2 (unfreeze SDE with small LR and update everything).

Now: only Phase 1: encoder + head adapt; SDE stays frozen at the meta-trained weights.

This leverages the shared meta-learned dynamics and avoids overfitting the SDE on two short trajectories, which is known to be unstable in stochastic differential equation models.
​
​

In [ ]:
# adaptation/few_shot_adapt.py
# HERO RUN: Data Efficiency Sweep (Granular)
# Measures performance as a function of available support steps.

import os
import copy
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

from config.base_config import cfg
from dataloaders.trajectory_datasets import TrajectoryDataset
from models.encoder import TrajEncoder
from models.neural_sde import NeuralSDE
from models.head import ForecastHead
from training.train_meta import simulate_neural_sde_batch

# --- CONFIG ---
N_SHOTS = 2

# Adaptation: only encoder + head, SDE frozen
ADAPT_STEPS = 50
LR_HEAD = 1e-2

# THE GRANULAR SWEEP
STEPS_SWEEP = [20, 40, 50, 80, 100, 120, 201]

def get_task_data(dataset, theta_id, device):
    rows = dataset.metadata[dataset.metadata["theta_id"] == theta_id]
    indices = rows.index.tolist()
    data = [dataset[i][0] for i in indices]
    return torch.stack(data).to(device)

def infer_z_star(encoder, support_trajs, obs_len):
    """Infer z from the OBSERVED PAST only."""
    encoder.eval()
    with torch.no_grad():
        obs = support_trajs[:, :obs_len, :]
        z_all = encoder(obs)
        z_mean = z_all.mean(dim=0, keepdim=True)
    return z_mean

def evaluate_model(sde, head, z, query_trajs, config, gen):
    """Evaluate on Query set (Full Future 0->T)."""
    sde.eval()
    head.eval()
    T = config.time_grid.T
    n_steps = config.time_grid.n_steps
    x_max = config.stability.max_state_abs

    with torch.no_grad():
        x0 = query_trajs[:, 0, :]
        z_expanded = z.expand(x0.size(0), -1)

        # Always simulate full duration for evaluation
        traj_pred = simulate_neural_sde_batch(
            sde, x0, z_expanded, T, n_steps, x_max, gen
        )
        x_T_pred = traj_pred[:, -1, :]
        final_pred = head(x_T_pred, z_expanded)

        mse_path = torch.mean((traj_pred - query_trajs) ** 2).item()
        mse_head = torch.mean((final_pred - query_trajs[:, -1, :]) ** 2).item()

    return mse_path, mse_head

def fine_tune_frozen_sde(encoder, sde, head, support_trajs, config, gen, limit_steps):
    """
    Few-shot adaptation on LIMITED history:
    - Slices support_trajs to [:limit_steps]
    - Freezes SDE
    - Adapts encoder + head using *final-step* MSE only
    """
    encoder_ft = copy.deepcopy(encoder)
    sde_ft = copy.deepcopy(sde)
    head_ft = copy.deepcopy(head)
    
    # 1. Prepare Training Data (Slice based on availability)
    train_target = support_trajs[:, :limit_steps, :]
    x0 = train_target[:, 0, :]
    target_final = train_target[:, -1, :]
    
    # 2. Calculate Simulation Params for this partial horizon
    T_full = config.time_grid.T
    n_steps_full = config.time_grid.n_steps
    dt = T_full / n_steps_full
    
    # limit_steps indices 0..limit_steps-1 --> limit_steps-1 steps to reach that time
    n_sim = limit_steps - 1
    T_train = dt * n_sim
    x_max = config.stability.max_state_abs
    
    # Encoder sees up to 50 steps max, or limit_steps if smaller
    enc_obs_len = min(limit_steps, 50)

    # Optimizer: encoder + head only
    param_group = list(encoder_ft.parameters()) + list(head_ft.parameters())
    optimizer = optim.Adam(param_group, lr=LR_HEAD)
    
    # Freeze SDE
    sde_ft.eval()
    for p in sde_ft.parameters():
        p.requires_grad = False

    encoder_ft.train()
    head_ft.train()
    
    for _ in range(ADAPT_STEPS):
        optimizer.zero_grad()
        
        # Infer z from observed prefix
        obs = train_target[:, :enc_obs_len, :]
        z = encoder_ft(obs).mean(dim=0, keepdim=True)
        z_expanded = z.expand(x0.size(0), -1)
        
        # Simulate partial path (SDE frozen)
        traj_pred = simulate_neural_sde_batch(
            sde_ft, x0, z_expanded, T_train, n_sim, x_max, gen
        )
        
        x_T = traj_pred[:, -1, :]
        pred = head_ft(x_T, z_expanded)
        
        # Final-step MSE only
        loss = nn.functional.mse_loss(pred, target_final)
        
        loss.backward()
        optimizer.step()
        
    return sde_ft, head_ft, z

def main():
    device = torch.device(cfg.device)
    print("🔬 Meta-Learning: Data Efficiency Sweep")
    print(f"Sweeping Step Limits: {STEPS_SWEEP}")
    print(f"Adaptation: {ADAPT_STEPS} steps (encoder + head), SDE frozen")

    ckpt_path = "checkpoints/meta_epoch_50.pt"
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found at {ckpt_path}. Run training first.")

    ckpt = torch.load(ckpt_path, map_location=device)
    x_dim = cfg.basis.x_dim
    z_dim = cfg.latent.latent_dim

    # Load Base Models
    encoder_base = TrajEncoder(x_dim, z_dim, cfg.latent.encoder_hidden_dim).to(device)
    sde_base = NeuralSDE(x_dim, z_dim, cfg.latent.sde_hidden_dim).to(device)
    head_base = ForecastHead(x_dim, z_dim, cfg.latent.head_hidden_dim).to(device)

    encoder_base.load_state_dict(ckpt["encoder"])
    sde_base.load_state_dict(ckpt["sde"])
    head_base.load_state_dict(ckpt["head"])

    gen = torch.Generator(device=device)
    gen.manual_seed(999)

    results_list = []
    index_path = os.path.join(cfg.paths.data_root, "index.csv")

    # Regimes Loop
    for regime in ["testA", "testB", "testC"]:
        print(f"\n🎯 Processing {regime}")
        try:
            ds_support = TrajectoryDataset(index_path, regime, "support", check_shapes=True)
            ds_query = TrajectoryDataset(index_path, regime, "query", check_shapes=True)
        except:
            continue

        # Tasks Loop
        tasks = ds_support.metadata["theta_id"].unique()
        for theta_id in tqdm(tasks, desc=regime):
            full_support = get_task_data(ds_support, theta_id, device)
            full_query = get_task_data(ds_query, theta_id, device)
            
            # --- THE SWEEP ---
            for limit_steps in STEPS_SWEEP:
                # 1. Zero-shot (uses only limit_steps for context)
                enc_obs_len = min(limit_steps, 50)
                z_star = infer_z_star(encoder_base, full_support[:N_SHOTS], enc_obs_len)
                zs_mse_path, zs_mse_head = evaluate_model(
                    sde_base, head_base, z_star, full_query, cfg, gen
                )

                # 2. Few-shot: encoder + head adaptation, SDE frozen
                sde_ft, head_ft, z_ft = fine_tune_frozen_sde(
                    encoder_base, sde_base, head_base, 
                    full_support[:N_SHOTS], cfg, gen, limit_steps
                )
                
                fs_mse_path, fs_mse_head = evaluate_model(
                    sde_ft, head_ft, z_ft, full_query, cfg, gen
                )

                results_list.append({
                    "regime": regime,
                    "theta_id": theta_id,
                    "steps_available": limit_steps,
                    "mse_head_zeroshot": zs_mse_head,
                    "mse_head_fewshot": fs_mse_head,
                    "mse_path_fewshot": fs_mse_path
                })

    # Save Results
    df = pd.DataFrame(results_list)
    out_path = "results/efficiency_sweep_results.csv"
    os.makedirs("results", exist_ok=True)
    df.to_csv(out_path, index=False)
    
    print("\n✅ Sweep Complete.")
    print("\n📊 Average Head MSE by Regime & Steps:")
    print(df.groupby(["regime", "steps_available"])[["mse_head_fewshot"]].mean())

if __name__ == "__main__":
    main()



📊 Average Head MSE by Regime & Steps:
                        mse_head_fewshot
regime steps_available                  
testA  20                       0.463751
       40                       0.416208
       50                       0.386411
       80                       0.370489
       100                      0.329901
       120                      0.308094
       201                      0.281341
testB  20                       1.904808
       40                       1.638203
       50                       1.491255
       80                       1.172230
       100                      0.962713
       120                      0.808459
       201                      0.600179
testC  20                       3.612789
       40                       3.002907
       50                       2.770425
       80                       2.071611
       100                      1.617800
       120                      1.255727
       201                      0.789732

In [ ]:
----------------------------------------------------

In [ ]:
NEW TECNIQUE ----- GATED INFERENCE 

In [ ]:
------------------------------------------------------------------

In [ ]:
first trial 

In [ ]:
#adaptation/gated_inference 
# adaptation/gated_inference.py
import os
import time
import copy
import torch
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pandas as pd
from tqdm import tqdm

from config.base_config import cfg
from dataloaders.trajectory_datasets import TrajectoryDataset
from models.encoder import TrajEncoder
from models.neural_sde import NeuralSDE
from models.head import ForecastHead
from training.train_meta import simulate_neural_sde_batch

# === HYPERPARAMETERS ===
K_SUBWINDOWS = 8        
MIN_WINDOW = 20         
MAX_TRAIN_CTX = 50      
GATE_ALPHA = 20.0       
GATE_TAU = 0.1          
N_SHOTS = 2
STEPS_SWEEP = [20, 40, 50, 80, 100, 120, 201] 
MC_SAMPLES = 5

def get_task_data(dataset, theta_id, device):
    rows = dataset.metadata[dataset.metadata["theta_id"] == theta_id]
    idx = rows.index.tolist()
    data = [dataset[i][0] for i in idx]
    return torch.stack(data).to(device)

def compute_latent_instability(encoder, support_trajs, k=K_SUBWINDOWS):
    """
    Measures instability by bootstrapping prefixes of varying lengths.
    Returns: (variance_scalar, mean_z)
    """
    B, T, D = support_trajs.shape
    effective_max_len = min(T, MAX_TRAIN_CTX)
    
    z_samples_list = []
    with torch.no_grad():
        for _ in range(k):
            # Sample random window length
            if effective_max_len > MIN_WINDOW:
                length = torch.randint(MIN_WINDOW, effective_max_len + 1, (1,)).item()
            else:
                length = min(T, MIN_WINDOW)
            
            batch_view = support_trajs[:, :length, :]
            z = encoder(batch_view)  # (B, z_dim)
            z_samples_list.append(z)
    
    # Stack: (K, B, z_dim)
    z_samples = torch.stack(z_samples_list, dim=0)
    
    # Compute variance per shot: (B, z_dim)
    z_mean = z_samples.mean(dim=0)  # (B, z_dim)
    z_diffs = z_samples - z_mean.unsqueeze(0)  # (K, B, z_dim)
    z_var = (z_diffs ** 2).mean(dim=0).sum(dim=-1)  # (B,) - variance per shot
    
    # Return scalar instability (mean across batch) and mean z
    instability = z_var.mean()  # scalar
    task_z = z_mean.mean(dim=0, keepdim=True)  # (1, z_dim)
    
    return instability, task_z

def compute_support_residual(sde, head, support, z, gen, cfg):
    """
    Computes residual error on the support set after adaptation.
    D̃_res = 1/N sum ||x_{t+1} - x̂_{t+1}(x_t, z)||²
    """
    device = support.device
    B, T, D = support.shape
    
    # Simulate from x_0
    T_full = cfg.time_grid.T
    n_steps_full = cfg.time_grid.n_steps
    dt = T_full / n_steps_full
    n_sim = min(T - 1, n_steps_full)
    T_sim = dt * n_sim
    x_max = cfg.stability.max_state_abs
    
    z_expanded = z.expand(B, -1)
    
    with torch.no_grad():
        traj_pred = simulate_neural_sde_batch(sde, support[:, 0], z_expanded, T_sim, n_sim, x_max, gen)
        # traj_pred: (B, n_sim+1, D)
        
        # Compare only the observed portion
        valid_len = min(traj_pred.shape[1], T)
        residual = F.mse_loss(traj_pred[:, :valid_len], support[:, :valid_len])
    
    return residual.item()

def adapt_head(head_init, sde, z_star, support, gen, cfg, n_adapt_steps=50, lr=1e-2):
    """
    Fine-tunes the head on the support set. Returns adapted head and wall-clock time.
    """
    start_time = time.time()
    
    head = copy.deepcopy(head_init)
    head.train()
    optimizer = optim.Adam(head.parameters(), lr=lr)
    
    B, T, D = support.shape
    z_expanded = z_star.expand(B, -1)
    
    T_full = cfg.time_grid.T
    n_steps_full = cfg.time_grid.n_steps
    dt = T_full / n_steps_full
    n_sim = T - 1
    T_sim = dt * n_sim
    x_max = cfg.stability.max_state_abs
    
    for _ in range(n_adapt_steps):
        optimizer.zero_grad()
        
        # Simulate SDE on support horizon
        traj = simulate_neural_sde_batch(sde, support[:, 0], z_expanded, T_sim, n_sim, x_max, gen)
        
        # Align lengths
        valid_len = min(traj.shape[1], T)
        traj_sliced = traj[:, :valid_len]
        target_sliced = support[:, :valid_len]
        
        # Head predicts final step
        pred = head(traj_sliced[:, -1], z_expanded)
        loss = F.mse_loss(pred, target_sliced[:, -1])
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(head.parameters(), 1.0)
        optimizer.step()
    
    adapt_time = time.time() - start_time
    head.eval()
    return head, adapt_time

def compute_onestep_mse(sde, support, z, cfg):
    """
    Computes One-Step MSE using SDE drift: x_{t+1} = x_t + f(x_t, z) * dt
    """
    device = support.device
    B, T, D = support.shape
    
    dt = cfg.time_grid.T / cfg.time_grid.n_steps
    
    # x_t, x_{t+1}
    x_t = support[:, :-1, :].reshape(-1, D)  # (B*(T-1), D)
    x_next = support[:, 1:, :].reshape(-1, D)  # (B*(T-1), D)
    
    # Expand z for each timestep
    z_expanded = z.expand(B, -1).repeat_interleave(T - 1, dim=0)  # (B*(T-1), z_dim)
    
    with torch.no_grad():
        # Get drift
        t_dummy = torch.zeros(x_t.shape[0], 1, device=device)
        drift = sde.f(t_dummy, x_t, z_expanded)  # (B*(T-1), D)
        
        # Euler step
        x_next_pred = x_t + drift * dt
        
        # MSE
        mse = F.mse_loss(x_next_pred, x_next).item()
    
    return mse

def gated_inference(encoder, sde, head, support, query, gen, cfg):
    """
    Complete gated inference pipeline with all metrics.
    """
    device = support.device
    
    # 1. Compute latent instability & z
    instability, task_z = compute_latent_instability(encoder, support)
    
    # 2. Compute support residual (for gate)
    support_residual = compute_support_residual(sde, head, support, task_z, gen, cfg)
    
    # 3. Compute gate: g = σ(α(τ - D̃))
    gate_input = GATE_ALPHA * (GATE_TAU - support_residual)
    g = torch.sigmoid(torch.tensor(gate_input)).item()
    
    # 4. Adapt head (measure time)
    head_adapted, adapt_time = adapt_head(head, sde, task_z, support, gen, cfg)
    
    # 5. Evaluate on query (full trajectory)
    B_q = query.shape[0]
    z_adapted = task_z.expand(B_q, -1)
    z_mean = torch.zeros_like(z_adapted)
    
    T_full = cfg.time_grid.T
    n_steps = cfg.time_grid.n_steps
    x_max = cfg.stability.max_state_abs
    
    # --- MC Sampling ---
    mc_trajs_adapted = []
    mc_trajs_mean = []
    
    with torch.no_grad():
        for _ in range(MC_SAMPLES):
            # Adapted trajectory
            traj_a = simulate_neural_sde_batch(sde, query[:, 0], z_adapted, T_full, n_steps, x_max, gen)
            # Mean trajectory (z=0)
            traj_m = simulate_neural_sde_batch(sde, query[:, 0], z_mean, T_full, n_steps, x_max, gen)
            
            mc_trajs_adapted.append(traj_a)
            mc_trajs_mean.append(traj_m)
    
    # Stack: (MC, B, T, D)
    mc_adapt = torch.stack(mc_trajs_adapted, dim=0)
    mc_mean = torch.stack(mc_trajs_mean, dim=0)
    
    # Gated mixture
    mc_gated = (1 - g) * mc_mean + g * mc_adapt  # (MC, B, T, D)
    
    # Compute statistics
    pred_mean = mc_gated.mean(dim=0)  # (B, T, D)
    pred_var = mc_gated.var(dim=0) + 1e-6  # (B, T, D)
    
    # --- Metrics ---
    mse_rollout = F.mse_loss(pred_mean, query).item()
    mse_final = F.mse_loss(pred_mean[:, -1], query[:, -1]).item()
    nll = F.gaussian_nll_loss(pred_mean, query, pred_var).item()
    
    # One-step check
    mse_onestep = compute_onestep_mse(sde, query, z_adapted, cfg)
    
    return {
        "gate": g,
        "z_instability": instability.item(),
        "support_residual": support_residual,
        "adapt_time": adapt_time,
        "mse_onestep": mse_onestep,
        "mse_rollout": mse_rollout,
        "mse_final": mse_final,
        "nll": nll
    }

def main():
    device = torch.device(cfg.device)
    print("🛡️  Gated Inference with Full Metrics...")
    
    ckpt_path = "checkpoints/meta_epoch_50.pt"
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")
    
    ckpt = torch.load(ckpt_path, map_location=device)
    x_dim, z_dim = cfg.basis.x_dim, cfg.latent.latent_dim
    
    encoder = TrajEncoder(x_dim, z_dim, cfg.latent.encoder_hidden_dim).to(device)
    sde = NeuralSDE(x_dim, z_dim, cfg.latent.sde_hidden_dim).to(device)
    head = ForecastHead(x_dim, z_dim, cfg.latent.head_hidden_dim).to(device)
    
    encoder.load_state_dict(ckpt['encoder'])
    sde.load_state_dict(ckpt['sde'])
    head.load_state_dict(ckpt['head'])
    
    encoder.eval()
    sde.eval()
    head.eval()
    
    gen = torch.Generator(device=device)
    gen.manual_seed(42)
    
    index_path = os.path.join(cfg.paths.data_root, "index.csv")
    results = []
    
    for regime in ["testA", "testB", "testC"]:
        print(f"\nProcessing {regime}...")
        try:
            ds_supp = TrajectoryDataset(index_path, regime, "support", check_shapes=True)
            ds_query = TrajectoryDataset(index_path, regime, "query", check_shapes=True)
        except Exception as e:
            print(f"Skipping {regime}: {e}")
            continue
        
        tasks = ds_supp.metadata["theta_id"].unique()
        
        for theta_id in tqdm(tasks, desc=regime):
            supp_full = get_task_data(ds_supp, theta_id, device)[:N_SHOTS]
            query = get_task_data(ds_query, theta_id, device)
            
            for steps in STEPS_SWEEP:
                supp_limited = supp_full[:, :steps, :]
                
                metrics = gated_inference(encoder, sde, head, supp_limited, query, gen, cfg)
                metrics["regime"] = regime
                metrics["steps_available"] = steps
                metrics["theta_id"] = theta_id
                
                results.append(metrics)
    
    df = pd.DataFrame(results)
    os.makedirs("results", exist_ok=True)
    out_path = "results/gated_metrics_full.csv"
    df.to_csv(out_path, index=False)
    
    print(f"\n✅ Saved to {out_path}")
    print("\n🏆 Summary (testC):")
    summary = df[df['regime'] == 'testC'].groupby('steps_available')[
        ['mse_rollout', 'mse_final', 'nll', 'adapt_time', 'gate']
    ].mean()
    print(summary)

if __name__ == "__main__":
    main()


In [ ]:
jovyan@jupyter-ec241027:~/DISSERTATION $ python -m analysis.show_full_table
📊 Loading results from results/gated_metrics_full.csv...

                        mse_rollout  mse_final            nll  adapt_time      gate
regime steps_available                                                             
testA  20                  0.149143   0.378122   61570.204818    2.271887  0.869322
       40                  0.141738   0.357380   58322.503646    4.625251  0.855980
       50                  0.138751   0.349367   56847.441276    5.628773  0.848882
       80                  0.139034   0.350843   57757.466536    8.661935  0.814503
       100                 0.137566   0.347538   57298.192383   10.813305  0.785126
       120                 0.138327   0.349810   58340.184961   12.885619  0.747877
       201                 0.146783   0.375384   63988.227344   21.661000  0.499819
testB  20                  0.532091   1.673432  215619.593099    2.254824  0.860605
       40                  0.480127   1.522114  195381.401693    4.681230  0.828904
       50                  0.464513   1.476056  189153.139844    5.691981  0.806155
       80                  0.496182   1.570400  209523.576172    8.877157  0.677338
       100                 0.551611   1.731952  237109.675130   11.043568  0.523673
       120                 0.598344   1.863592  255588.288672   13.130956  0.368940
       201                 0.692839   2.120925  280981.050521   21.608616  0.052334
testC  20                  0.999654   3.154033  400710.003125    2.278457  0.847858
       40                  0.930326   2.938221  373992.618229    4.644393  0.767608
       50                  0.942525   2.969918  385404.646354    5.680049  0.708179
       80                  1.101093   3.432198  469460.127083    8.918832  0.442353
       100                 1.219467   3.769929  512206.479687   11.056515  0.254982
       120                 1.289614   3.968820  533209.270312   12.977464  0.137453
       201                 1.359028   4.161782  551332.208333   21.667891  0.004155

✅ Done. This table contains all your experimental results.
jovyan@jupyter-ec241027:~/DISSERTATION $ 

In [ ]:
GATED FINE TUNING 

In [ ]:
# gated_finetuning 
import os
import time
import copy
import torch
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pandas as pd
from tqdm import tqdm

from config.base_config import cfg
from dataloaders.trajectory_datasets import TrajectoryDataset
from models.encoder import TrajEncoder
from models.neural_sde import NeuralSDE
from models.head import ForecastHead
from training.train_meta import simulate_neural_sde_batch

# === HYPERPARAMETERS ===
# Optimization (Plasticity)
ADAPT_STEPS = 50        # How many gradient steps to take
LR_Z = 1e-2             # Learning rate for Latent Code z
LR_HEAD = 1e-2          # Learning rate for Forecast Head
N_SHOTS = 2             # Number of trajectories used for adaptation

# Gating (Safety)
GATE_ALPHA = 20.0       # Steepness of the gate sigmoid [cite: 43]
GATE_TAU = 0.05         # Error threshold (If residual > 0.05, close the gate) [cite: 43]
MC_SAMPLES = 5          # Number of SDE samples for NLL/Uncertainty [cite: 16]

# The Sweep
STEPS_SWEEP = [20, 40, 50, 80, 100, 120, 201]

def get_task_data(dataset, theta_id, device):
    rows = dataset.metadata[dataset.metadata["theta_id"] == theta_id]
    idx = rows.index.tolist()
    data = [dataset[i][0] for i in idx]
    return torch.stack(data).to(device)

def adapt_model(sde, head_init, z_init, support, gen, cfg):
    """
    Phase 1: Adaptation (Plasticity).
    Optimizes z and Head to fit the support set.
    """
    start_time = time.time()
    
    # Clone models to avoid overwriting base
    head = copy.deepcopy(head_init)
    head.train()
    
    # Clone Z and enable gradients (Latent Adaptation)
    z_adapted = z_init.clone().detach()
    z_adapted.requires_grad = True
    
    # Optimizer targeting z and head
    optimizer = optim.Adam([
        {'params': head.parameters(), 'lr': LR_HEAD},
        {'params': [z_adapted], 'lr': LR_Z}
    ])
    
    # SDE is frozen (we only adapt parameters z, not weights theta)
    for p in sde.parameters():
        p.requires_grad = False
        
    B, T, D = support.shape
    
    # Simulation Parameters
    T_full = cfg.time_grid.T
    n_steps_full = cfg.time_grid.n_steps
    dt = T_full / n_steps_full
    n_sim = T - 1
    T_sim = dt * n_sim
    x_max = cfg.stability.max_state_abs
    
    for _ in range(ADAPT_STEPS):
        optimizer.zero_grad()
        
        # Expand z for batch
        z_batch = z_adapted.expand(B, -1)
        
        # 1. Simulate dynamics (Physics Match)
        traj = simulate_neural_sde_batch(sde, support[:, 0], z_batch, T_sim, n_sim, x_max, gen)
        
        # 2. Slice to match valid data length
        valid_len = min(traj.shape[1], T)
        traj_sliced = traj[:, :valid_len]
        target_sliced = support[:, :valid_len]
        
        # Loss 1: Path Reconstruction (Physics)
        loss_path = F.mse_loss(traj_sliced, target_sliced)
        
        # Loss 2: Head Prediction (Goal)
        pred_head = head(traj_sliced[:, -1], z_batch)
        loss_head = F.mse_loss(pred_head, target_sliced[:, -1])
        
        # Combined objective
        total_loss = loss_path + loss_head
        
        total_loss.backward()
        optimizer.step()
        
    adapt_time = time.time() - start_time
    head.eval()
    
    return head, z_adapted.detach(), adapt_time

def compute_residual(sde, head, z, support, gen, cfg):
    """
    Phase 2: The Sensor.
    Computes D_res (Eq 7): Residual error of the adapted model on support data.
    """
    B, T, D = support.shape
    T_full = cfg.time_grid.T
    n_steps_full = cfg.time_grid.n_steps
    dt = T_full / n_steps_full
    n_sim = min(T - 1, n_steps_full)
    T_sim = dt * n_sim
    x_max = cfg.stability.max_state_abs
    
    z_expanded = z.expand(B, -1)
    
    with torch.no_grad():
        # Simulate using the *Adapted* Z
        traj_pred = simulate_neural_sde_batch(sde, support[:, 0], z_expanded, T_sim, n_sim, x_max, gen)
        valid_len = min(traj_pred.shape[1], T)
        
        # Calculate MSE (Residual) 
        residual = F.mse_loss(traj_pred[:, :valid_len], support[:, :valid_len])
        return residual.item()

def gated_inference(encoder, sde, head, support, query, gen, cfg):
    """
    Phase 3: Gated Inference.
    Combines Safe (Mean) and Smart (Adapted) predictions based on Residual.
    """
    # 1. Initial Z (Zero-Shot)
    with torch.no_grad():
        # Use prefix of length up to 50 for initial encoding
        enc_len = min(support.shape[1], 50)
        z_init = encoder(support[:, :enc_len]).mean(dim=0, keepdim=True)

    # 2. Run Adaptation (Plasticity)
    head_opt, z_opt, adapt_time = adapt_model(sde, head, z_init, support, gen, cfg)
    
    # 3. Compute Residual (The Sensor) on Support Set
    d_res = compute_residual(sde, head_opt, z_opt, support, gen, cfg)
    
    # 4. Compute Gate (The Safety Switch) [cite: 43]
    # g = sigmoid( alpha * (tau - residual) )
    # If residual is LOW (good fit), gate -> 1.
    # If residual is HIGH (bad fit), gate -> 0.
    g = torch.sigmoid(torch.tensor(GATE_ALPHA * (GATE_TAU - d_res))).item()
    
    # 5. Predictions (Mixing)
    B_q = query.shape[0]
    z_smart = z_opt.expand(B_q, -1)
    z_safe = torch.zeros_like(z_smart) # Mean physics (z=0)
    
    T_full = cfg.time_grid.T
    n_steps = cfg.time_grid.n_steps
    x_max = cfg.stability.max_state_abs
    
    mc_preds = []
    
    with torch.no_grad():
        for _ in range(MC_SAMPLES):
            # Path A: Adapted (Smart)
            traj_smart = simulate_neural_sde_batch(sde, query[:, 0], z_smart, T_full, n_steps, x_max, gen)
            
            # Path B: Mean (Safe)
            traj_safe = simulate_neural_sde_batch(sde, query[:, 0], z_safe, T_full, n_steps, x_max, gen)
            
            # Weighted Mixture [cite: 2]
            traj_mix = (1 - g) * traj_safe + g * traj_smart
            mc_preds.append(traj_mix)
            
    # Aggregate MC Samples
    mc_tensor = torch.stack(mc_preds, dim=0)
    pred_mean = mc_tensor.mean(dim=0)
    pred_var = mc_tensor.var(dim=0) + 1e-6
    
    # Metrics
    mse_rollout = F.mse_loss(pred_mean, query).item()
    mse_final = F.mse_loss(pred_mean[:, -1], query[:, -1]).item()
    nll = F.gaussian_nll_loss(pred_mean, query, pred_var).item()
    
    return {
        "gate_value": g,
        "residual_error": d_res,
        "adapt_time": adapt_time,
        "mse_rollout": mse_rollout,
        "mse_final": mse_final,
        "nll": nll
    }

def main():
    device = torch.device(cfg.device)
    print("🛡️ Running Gated Fine-Tuning (Plasticity + Safety)...")
    
    ckpt_path = "checkpoints/meta_epoch_50.pt"
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")
        
    ckpt = torch.load(ckpt_path, map_location=device)
    x_dim, z_dim = cfg.basis.x_dim, cfg.latent.latent_dim
    
    encoder = TrajEncoder(x_dim, z_dim, cfg.latent.encoder_hidden_dim).to(device)
    sde = NeuralSDE(x_dim, z_dim, cfg.latent.sde_hidden_dim).to(device)
    head = ForecastHead(x_dim, z_dim, cfg.latent.head_hidden_dim).to(device)
    
    encoder.load_state_dict(ckpt['encoder'])
    sde.load_state_dict(ckpt['sde'])
    head.load_state_dict(ckpt['head'])
    
    encoder.eval(); sde.eval(); head.eval()
    gen = torch.Generator(device=device)
    gen.manual_seed(42)
    
    index_path = os.path.join(cfg.paths.data_root, "index.csv")
    results = []
    
    for regime in ["testA", "testB", "testC"]:
        print(f"\nProcessing {regime}...")
        try:
            ds_supp = TrajectoryDataset(index_path, regime, "support")
            ds_query = TrajectoryDataset(index_path, regime, "query")
        except: continue
        
        tasks = ds_supp.metadata["theta_id"].unique()
        
        for theta_id in tqdm(tasks, desc=regime):
            supp_full = get_task_data(ds_supp, theta_id, device)[:N_SHOTS]
            query = get_task_data(ds_query, theta_id, device)
            
            for steps in STEPS_SWEEP:
                supp_limited = supp_full[:, :steps, :]
                
                metrics = gated_inference(
                    encoder, sde, head, supp_limited, query, gen, cfg
                )
                
                metrics["regime"] = regime
                metrics["steps_available"] = steps
                results.append(metrics)
                
    df = pd.DataFrame(results)
    os.makedirs("results", exist_ok=True)
    df.to_csv("results/gated_finetuning_full.csv", index=False)
    
    print("\n✅ Results saved to results/gated_finetuning_full.csv")
    print("\n🏆 Final Summary (Test C):")
    print(df[df['regime'] == 'testC'].groupby('steps_available')[
        ['mse_rollout', 'residual_error', 'gate_value']
    ].mean())

if __name__ == "__main__":
    main()

In [ ]:
jovyan@jupyter-ec241027:~/DISSERTATION $ python -c "import pandas as pd; pd.set_option('display.max_rows', None); pd.set_option('display.max_columns', None); pd.set_option('display.width', 1000); df = pd.read_csv('results/gated_finetuning_full.csv'); print(df.groupby(['regime', 'steps_available'])[['mse_rollout', 'mse_final', 'nll', 'residual_error', 'gate_value', 'adapt_time']].mean())"
                        mse_rollout  mse_final            nll  residual_error  gate_value  adapt_time
regime steps_available                                                                               
testA  20                  0.148611   0.378782   64288.720052        0.004934    0.711147    1.942467
       40                  0.135504   0.342769   57959.016211        0.009707    0.690930    3.923541
       50                  0.130326   0.328209   55551.559570        0.011957    0.681117    4.869926
       80                  0.128997   0.325384   55382.371094        0.020076    0.644333    7.494756
       100                 0.127936   0.322630   55095.879167        0.026369    0.614664    9.369567
       120                 0.127462   0.321512   55189.393815        0.033034    0.582728   11.299527
       201                 0.135335   0.343708   58957.621094        0.068787    0.417205   18.395092
testB  20                  0.550195   1.730781  235300.490234        0.008500    0.696093    1.909386
       40                  0.477509   1.515005  202714.852214        0.018803    0.650231    3.955121
       50                  0.456840   1.448067  192680.652083        0.024840    0.622018    4.820174
       80                  0.492631   1.554103  211643.250781        0.048809    0.506848    7.464645
       100                 0.531548   1.666460  227992.109505        0.071077    0.407917    9.307756
       120                 0.574086   1.787969  243906.487630        0.098833    0.308938   11.032620
       201                 0.682249   2.090992  277625.694271        0.295621    0.065723   18.320789
testC  20                  1.054583   3.317822  448663.517708        0.013358    0.674757    1.803848
       40                  0.974468   3.058911  410106.342187        0.035734    0.570174    3.827896
       50                  0.992151   3.101888  418394.844271        0.049201    0.507466    4.743303
       80                  1.139920   3.533765  484460.578125        0.100419    0.301274    7.280989
       100                 1.231033   3.797108  514005.093229        0.146502    0.184342    8.991279
       120                 1.290766   3.968572  532686.786458        0.205938    0.106679   10.744542
       201                 1.357737   4.158102  550954.443750        0.624177    0.005941   17.888941
jovyan@jupyter-ec241027:~/DISSERTATION $ 

In [ ]:
NEW VERSIO GATED FINE REGULARIZED TUNING CODE SEE DESCRIPTION BELOW

In [ ]:
NEW VERSION 
This is a very smart move. You are right to pause—rushing into a "kitchen sink" implementation without understanding the components is exactly why the previous script felt disconnected from your earlier success.

Here is the breakdown of your Professor's note (`cecilia_notes.pdf`), translated into plain English and Physics logic.

### 1. Suggestion 1: The "Safety Gate" (The Mixture)

**What it says:**
Instead of trusting your Meta-SDE 100% of the time, you should use a weighted average of a "Safe Model" (Mean Physics) and your "Smart Model" (Adapted Physics).

* 
**The Equation:** .


* **The Logic:**
* If : You panic and revert to the average (like the GRU does). Safe but boring.
* If : You trust your adaptation. High performance.


* **Why we need it:** This solves the crash on **Test C (20 steps)**. When the model is confused, the gate forces it to be safe, preventing the massive 3.6 error.

### 2. Suggestion 2: The "Sensor" (Instability Score)

**What it says:**
How do we decide the value of ? Do not rely on a single guess.

* 
**The Method:** "Infer latent codes on multiple random support sub-windows".


* 
**The Metric:** Calculate the **Variance** of these  vectors.


* If  stays the same no matter which window you look at  **Stable**  High Confidence ().
* If  jumps around wildly  **Unstable**  Low Confidence ().




* **Why we need it:** This is the *trigger* for the Safety Gate. It detects when the physics are "Out-Of-Distribution" without needing ground truth labels.

### 3. Suggestion 3: The "Governor" (Latent Tempering)

**What it says:**
Prevent the model from hallucinating extreme physics from limited data.

* 
**The Method:** "Latent Tempering".


* 
**The Equation:**  where .


* **The Logic:** Mechanically shrink the  vector towards 0 (the mean).
* **Why we need it:** This is a simpler alternative to Gating. If the error is high, just dampen the adaptation.



---

### 🛑 The Conflict & The Solution

There is a subtle conflict between your **Old Script (`few_shot_adapt.py`)** and **Suggestion 1**.

* **Old Script:** "I will use Gradient Descent to force  to fit the data, even if it moves far from the prior." (Aggressive Plasticity).
* **Suggestion 1:** "If  looks unstable or error is high, ignore it and use the mean." (Conservative Safety).

**We need BOTH.**
If we only do Suggestion 1 (Zero-Shot Gating), we lose the "Plasticity" that allows you to beat the GRU on Test A.
If we only do the Old Script, we crash on Test C.

### 💡 The Implementation Plan

We should implement **Suggestions 1 and 2 together**, wrapped around the optimization from your Old Script.

**Step-by-Step Logic for the Code:**

1. **Optimization (Plasticity):** Run Gradient Descent on  (just like `few_shot_adapt.py`) to try and find the best physics.
2. 
**Suggestion 2 (The Sensor):** Measure the *residual error* of this optimized  on the support set.


* 
*Note:* The professor lists "Residual Error"  as a primary input for the gate. This is better than variance for fine-tuned models.




3. 
**Suggestion 1 (The Gate):** Calculate .


* If optimization worked (Low Error)    Use the Optimized .
* If optimization failed (High Error, Test C)    Throw away the Optimized , use Mean Physics.



**Decision:**
Do you agree with this hybrid approach? It satisfies the Professor's demand for safety mechanisms  while preserving the fine-tuning capability that gave you your best results previously.

If yes, I will provide the script that implements exactly this pipeline.

In [ ]:
# adaptation/gated_finetuning_regularized .py
import os
import time
import copy
import torch
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pandas as pd
from tqdm import tqdm

from config.base_config import cfg
from dataloaders.trajectory_datasets import TrajectoryDataset
from models.encoder import TrajEncoder
from models.neural_sde import NeuralSDE
from models.head import ForecastHead
from training.train_meta import simulate_neural_sde_batch

# === HYPERPARAMETERS ===
ADAPT_STEPS = 50        
LR_Z = 1e-2             
LR_HEAD = 1e-2          
N_SHOTS = 2             

# Safety / Regularization
BETA_REG = 0.01         # Suggestion 3 (Regularization Weight)
GATE_ALPHA = 20.0       
GATE_TAU = 0.05         
MC_SAMPLES = 5          

STEPS_SWEEP = [20, 40, 50, 80, 100, 120, 201]
RESULTS_PATH = "results/gated_regularized_final.csv"
SAVE_EVERY = 5

def get_task_data(dataset, theta_id, device):
    rows = dataset.metadata[dataset.metadata["theta_id"] == theta_id]
    idx = rows.index.tolist()
    data = [dataset[i][0] for i in idx]
    return torch.stack(data).to(device)

def adapt_model(sde, head_init, z_init, support, gen, cfg):
    start_time = time.time()
    head = copy.deepcopy(head_init); head.train()
    z_adapted = z_init.clone().detach(); z_adapted.requires_grad = True
    
    optimizer = optim.Adam([
        {'params': head.parameters(), 'lr': LR_HEAD},
        {'params': [z_adapted], 'lr': LR_Z}
    ])
    
    for p in sde.parameters(): p.requires_grad = False
        
    B, T, D = support.shape
    T_full = cfg.time_grid.T; n_steps = cfg.time_grid.n_steps
    dt = T_full / n_steps
    n_sim = T - 1; T_sim = dt * n_sim
    x_max = cfg.stability.max_state_abs
    
    for _ in range(ADAPT_STEPS):
        optimizer.zero_grad()
        z_batch = z_adapted.expand(B, -1)
        
        # Simulate
        traj = simulate_neural_sde_batch(sde, support[:, 0], z_batch, T_sim, n_sim, x_max, gen)
        valid_len = min(traj.shape[1], T)
        
        # Correct Slicing for Loss
        traj_slice = traj[:, :valid_len, :]        # (B, L, D)
        supp_slice = support[:, :valid_len, :]     # (B, L, D)
        
        # 1. Path Loss (Physics)
        loss_path = F.mse_loss(traj_slice, supp_slice)
        
        # 2. Head Loss (Forecast)
        # We predict using the FINAL state of the simulation slice
        final_state_pred = traj_slice[:, -1, :]    # (B, D)
        final_state_target = supp_slice[:, -1, :]  # (B, D)
        
        head_pred = head(final_state_pred, z_batch)
        loss_head = F.mse_loss(head_pred, final_state_target)
        
        # 3. Regularization (Suggestion 3)
        loss_reg = BETA_REG * torch.sum(z_adapted ** 2)
        
        total_loss = loss_path + loss_head + loss_reg
        
        total_loss.backward()
        optimizer.step()
        
    return head, z_adapted.detach(), time.time() - start_time

def compute_residual(sde, head, z, support, gen, cfg):
    B, T, D = support.shape
    T_full = cfg.time_grid.T; n_steps = cfg.time_grid.n_steps
    dt = T_full / n_steps
    n_sim = min(T - 1, n_steps); T_sim = dt * n_sim
    x_max = cfg.stability.max_state_abs
    z_exp = z.expand(B, -1)
    
    with torch.no_grad():
        traj = simulate_neural_sde_batch(sde, support[:, 0], z_exp, T_sim, n_sim, x_max, gen)
        valid_len = min(traj.shape[1], T)
        return F.mse_loss(traj[:, :valid_len], support[:, :valid_len]).item()

def gated_inference(encoder, sde, head, support, query, gen, cfg):
    # 1. Init
    with torch.no_grad():
        enc_len = min(support.shape[1], 50)
        z_init = encoder(support[:, :enc_len]).mean(dim=0, keepdim=True)

    # 2. Adapt (Regularized)
    head_opt, z_opt, adapt_time = adapt_model(sde, head, z_init, support, gen, cfg)
    
    # 3. Gate
    d_res = compute_residual(sde, head_opt, z_opt, support, gen, cfg)
    g = torch.sigmoid(torch.tensor(GATE_ALPHA * (GATE_TAU - d_res))).item()
    
    # 4. Predict
    B_q = query.shape[0]
    z_smart = z_opt.expand(B_q, -1); z_safe = torch.zeros_like(z_smart)
    T_full = cfg.time_grid.T; n_steps = cfg.time_grid.n_steps
    x_max = cfg.stability.max_state_abs
    
    mc_preds = []
    with torch.no_grad():
        for _ in range(MC_SAMPLES):
            t_smart = simulate_neural_sde_batch(sde, query[:, 0], z_smart, T_full, n_steps, x_max, gen)
            t_safe = simulate_neural_sde_batch(sde, query[:, 0], z_safe, T_full, n_steps, x_max, gen)
            mc_preds.append((1 - g) * t_safe + g * t_smart)
            
    mc_tensor = torch.stack(mc_preds, dim=0)
    mean = mc_tensor.mean(dim=0); var = mc_tensor.var(dim=0) + 1e-6
    
    return {
        "gate_value": g, "residual_error": d_res, "adapt_time": adapt_time,
        "mse_rollout": F.mse_loss(mean, query).item(),
        "mse_final": F.mse_loss(mean[:, -1], query[:, -1]).item(),
        "nll": F.gaussian_nll_loss(mean, query, var).item()
    }

def main():
    device = torch.device(cfg.device)
    print("🛡️  Resumable Gated Finetuning (REGULARIZED + FIXED) Started...")
    
    ckpt = torch.load("checkpoints/meta_epoch_50.pt", map_location=device)
    x_dim, z_dim = cfg.basis.x_dim, cfg.latent.latent_dim
    encoder = TrajEncoder(x_dim, z_dim, cfg.latent.encoder_hidden_dim).to(device)
    sde = NeuralSDE(x_dim, z_dim, cfg.latent.sde_hidden_dim).to(device)
    head = ForecastHead(x_dim, z_dim, cfg.latent.head_hidden_dim).to(device)
    encoder.load_state_dict(ckpt['encoder']); sde.load_state_dict(ckpt['sde']); head.load_state_dict(ckpt['head'])
    encoder.eval(); sde.eval(); head.eval()
    
    gen = torch.Generator(device=device); gen.manual_seed(42)
    index_path = os.path.join(cfg.paths.data_root, "index.csv")

    completed_keys = set()
    if os.path.exists(RESULTS_PATH):
        print(f"Resuming from {RESULTS_PATH}...")
        try:
            for _, row in pd.read_csv(RESULTS_PATH).iterrows():
                completed_keys.add(f"{row['regime']}_{row['theta_id']}_{int(row['steps_available'])}")
        except: pass
    else:
        pd.DataFrame(columns=["regime", "theta_id", "steps_available", 
                              "gate_value", "residual_error", "adapt_time", 
                              "mse_rollout", "mse_final", "nll"]).to_csv(RESULTS_PATH, index=False)

    buffer = []
    
    for regime in ["testA", "testB", "testC"]:
        try:
            ds_supp = TrajectoryDataset(index_path, regime, "support")
            ds_query = TrajectoryDataset(index_path, regime, "query")
        except: continue
        
        tasks = ds_supp.metadata["theta_id"].unique()
        
        for theta_id in tqdm(tasks, desc=regime):
            needed = False
            for steps in STEPS_SWEEP:
                if f"{regime}_{theta_id}_{steps}" not in completed_keys: needed = True; break
            if not needed: continue

            supp_full = get_task_data(ds_supp, theta_id, device)[:N_SHOTS]
            query = get_task_data(ds_query, theta_id, device)
            
            for steps in STEPS_SWEEP:
                key = f"{regime}_{theta_id}_{steps}"
                if key in completed_keys: continue
                
                metrics = gated_inference(encoder, sde, head, supp_full[:, :steps], query, gen, cfg)
                metrics.update({"regime": regime, "theta_id": theta_id, "steps_available": steps})
                buffer.append(metrics)
                
            if len(buffer) >= SAVE_EVERY:
                pd.DataFrame(buffer).to_csv(RESULTS_PATH, mode='a', header=False, index=False)
                buffer = []
                
    if buffer: pd.DataFrame(buffer).to_csv(RESULTS_PATH, mode='a', header=False, index=False)
        
    print("\n✅ Regularized Run Complete.")
    full_df = pd.read_csv(RESULTS_PATH)
    print(full_df[full_df['regime']=='testC'].groupby('steps_available')[['mse_rollout', 'residual_error', 'gate_value']].mean())

if __name__ == "__main__":
    main()

In [ ]:
jovyan@jupyter-ec241027:~/DISSERTATION $ python -c "import pandas as pd; pd.set_option('display.max_rows', None); pd.set_option('display.width', 1000); cols = ['gate_value', 'residual_error', 'adapt_time', 'mse_rollout', 'mse_final', 'nll', 'regime', 'theta_id', 'steps_available']; df = pd.read_csv('results/gated_regularized_final.csv', header=0, names=cols); print(df.groupby(['regime', 'steps_available'])[['mse_rollout', 'mse_final', 'gate_value', 'residual_error']].mean())"
                        mse_rollout  mse_final  gate_value  residual_error
regime steps_available                                                    
testA  20                  0.163024   0.418012    0.705765        0.006234
       40                  0.151176   0.386648    0.678421        0.012600
       50                  0.145006   0.369844    0.667150        0.015133
       80                  0.143236   0.365653    0.619306        0.025480
       100                 0.141787   0.361896    0.584738        0.032676
       120                 0.140469   0.358335    0.549244        0.039998
       201                 0.143582   0.366713    0.384999        0.075926
testB  20                  0.615883   1.916970    0.688535        0.010271
       40                  0.537574   1.696900    0.634603        0.022228
       50                  0.508573   1.607941    0.603291        0.028832
       80                  0.546805   1.718205    0.462052        0.058223
       100                 0.580930   1.814311    0.353098        0.083940
       120                 0.615714   1.910909    0.253264        0.114775
       201                 0.687981   2.107191    0.052072        0.314095
testC  20                  1.164332   3.638441    0.666591        0.015212
       40                  1.040159   3.269007    0.556063        0.038704
       50                  1.041468   3.261224    0.491407        0.052635
       80                  1.188235   3.682240    0.268041        0.110040
       100                 1.267871   3.907871    0.151928        0.161313
       120                 1.313557   4.036008    0.082833        0.224730
       201                 1.358913   4.161439    0.004033        0.647107

In [ ]:
jovyan@jupyter-ec241027:~/DISSERTATION $ python -c "import pandas as pd; pd.set_option('display.max_rows', None); pd.set_option('display.max_columns', None); pd.set_option('display.width', 1000); df = pd.read_csv('results/gated_regularized_final_fixed.csv'); print(df.groupby(['regime', 'steps_available'])[['mse_rollout', 'mse_final', 'gate_value', 'residual_error', 'adapt_time']].mean())"
                        mse_rollout  mse_final  gate_value  residual_error  adapt_time
regime steps_available                                                                
testA  20                  0.163024   0.418012    0.705765        0.006234    1.817753
       40                  0.151176   0.386648    0.678421        0.012600    3.793741
       50                  0.145006   0.369844    0.667150        0.015133    4.670079
       80                  0.143236   0.365653    0.619306        0.025480    7.184729
       100                 0.141787   0.361896    0.584738        0.032676    8.848075
       120                 0.140469   0.358335    0.549244        0.039998   10.538971
       201                 0.143582   0.366713    0.384999        0.075926   17.468959
testB  20                  0.615883   1.916970    0.688535        0.010271    1.777138
       40                  0.537574   1.696900    0.634603        0.022228    3.760319
       50                  0.508573   1.607941    0.603291        0.028832    4.648859
       80                  0.546805   1.718205    0.462052        0.058223    7.145069
       100                 0.580930   1.814311    0.353098        0.083940    8.845493
       120                 0.615714   1.910909    0.253264        0.114775   10.554549
       201                 0.687981   2.107191    0.052072        0.314095   17.457668
testC  20                  1.164332   3.638441    0.666591        0.015212    1.768608
       40                  1.040159   3.269007    0.556063        0.038704    3.787049
       50                  1.041468   3.261224    0.491407        0.052635    4.651022
       80                  1.188235   3.682240    0.268041        0.110040    7.138117
       100                 1.267871   3.907871    0.151928        0.161313    8.857251
       120                 1.313557   4.036008    0.082833        0.224730   10.550450
       201                 1.358913   4.161439    0.004033        0.647107   17.477302